## IMAGES

### Check for Georeferenced Images (Requires Input path)


In [1]:
import os
import re
import warnings
import csv
from pathlib import Path
from collections import defaultdict
import win32com.client
import pythoncom
import rasterio
import numpy as np

warnings.filterwarnings('ignore')

class GeorefImageProcessor:
    def __init__(self, source_path, output_dir):
        """
        Initialize the GeorefImageProcessor.
        
        Args:
            source_path: Path to scan for shortcut files
            output_dir: Directory to save output CSV
        """
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.output_file = "Raster_ImageExtent.csv"
        
        # Data storage
        self.shortcuts = []
        self.shortcuts_with_targets = []
        self.raster_data = []
        self.all_images = []
        
        print(f"Initialized GeorefImageProcessor")
        print(f"Source path: {self.source_path}")
        print(f"Output directory: {self.output_dir}")

    def parse_image_info(self, filename):
        """Parse filename pattern and extract components."""
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').replace('_Finalized', '')
        parts = [p.strip() for p in filename.split('#') if p.strip()]
        
        # Initialize variables
        image_num = flight_num = sortie = bestell_id = date = None
        
        if parts:
            image_num = parts[0]
        
        # Extract Bestell ID (numbers only)
        for part in parts:
            if part.startswith('ID') and any(c.isdigit() for c in part[2:]):
                # Extract only numbers from BestellID
                numbers = re.findall(r'\d+', part)
                bestell_id = numbers[0] if numbers else None
                break
        
        # Find date
        for part in parts:
            if re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                date = part
                break
        
        # Find flight number and SORTIE
        if len(parts) >= 3:
            for part in parts[1:-1]:
                if part != bestell_id and not re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                    if not flight_num:
                        flight_num = part
                    elif not sortie:
                        sortie = part
        
        return {
            'image_number': image_num,
            'flight_number': flight_num,
            'sortie': sortie,
            'bestell_id': bestell_id,
            'date': date,
            'original_name': filename
        }

    def find_and_process_shortcuts(self):
        """Find all .lnk files and parse their information."""
        print(f"\nScanning for shortcut files in {self.source_path}...")
        
        for root, dirs, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    file_without_ext = file[:-4]
                    parsed_info = self.parse_image_info(file_without_ext)
                    parsed_info['full_path'] = Path(root) / file
                    parsed_info['folder'] = root
                    self.shortcuts.append(parsed_info)
        
        print(f"Found {len(self.shortcuts)} shortcut files")
        return self.shortcuts

    def get_shortcut_target(self, lnk_path):
        """Extract target path from Windows shortcut."""
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target_path = shortcut.TargetPath
            pythoncom.CoUninitialize()
            return target_path if target_path else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path}: {e}")
            return None

    def find_modified_versions(self, original_path):
        """Look for modified versions of an original image."""
        if not original_path or not original_path.exists():
            return []
        
        folder = original_path.parent
        stem = original_path.stem
        modified_patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        modified_files = []
        
        # Check for files with common modification suffixes
        for pattern in modified_patterns:
            potential_file = folder / f"{stem}{pattern}{original_path.suffix}"
            if potential_file.exists():
                modified_files.append(potential_file)
        
        # Check for other files with modification patterns
        for file in folder.iterdir():
            if file.is_file() and original_path.suffix == file.suffix:
                file_stem = file.stem
                if (file_stem.startswith(stem) and 
                    any(pattern in file_stem for pattern in modified_patterns)):
                    modified_files.append(file)
        
        return list(set(modified_files))

    def extract_shortcut_targets(self):
        """Extract target paths from all shortcuts and find modified versions."""
        print("\nExtracting shortcut targets and finding modified versions...")
        
        for shortcut_data in self.shortcuts:
            lnk_path = shortcut_data['full_path']
            target_path = self.get_shortcut_target(lnk_path)
            
            if target_path:
                target_path = Path(target_path)
                shortcut_data.update({
                    'target_path': target_path,
                    'target_exists': target_path.exists(),
                    'target_folder': target_path.parent,
                    'target_filename': target_path.name
                })
                
                # Parse target filename
                target_info = self.parse_image_info(target_path.stem)
                shortcut_data.update({
                    'target_image_number': target_info['image_number'],
                    'target_bestell_id': target_info['bestell_id'],
                    'target_sortie': target_info['sortie'],
                    'target_date': target_info['date']
                })
                
                # Find modified versions
                if shortcut_data['target_exists']:
                    modified_files = self.find_modified_versions(target_path)
                    if modified_files:
                        shortcut_data['modified_files'] = modified_files
                
                self.shortcuts_with_targets.append(shortcut_data)
        
        print(f"Successfully processed {len(self.shortcuts_with_targets)} shortcuts")
        return self.shortcuts_with_targets

    def calculate_tiff_area_extent(self, tiff_path):
        """Calculate area and geographic extent of TIFF file."""
        try:
            with rasterio.open(tiff_path) as src:
                rows, cols = src.height, src.width
                transform = src.transform
                pixel_res_x, pixel_res_y = transform.a, -transform.e
                bounds = src.bounds
                
                pixel_area = abs(pixel_res_x * pixel_res_y)
                total_area = pixel_area * rows * cols
                
                return {
                    'area_m2': float(total_area),
                    'rows': int(rows),
                    'cols': int(cols),
                    'pixel_res_x_m': float(pixel_res_x),
                    'pixel_res_y_m': float(pixel_res_y),
                    'bounds': bounds,
                    'is_georeferenced': src.crs is not None
                }
        except Exception as e:
            print(f"Error processing TIFF {tiff_path.name}: {e}")
            return None

    def process_georeferenced_images(self):
        """Process all georeferenced TIFF files and collect data."""
        print("\nProcessing georeferenced TIFF files...")
        
        for shortcut in self.shortcuts_with_targets:
            # Create base record for original image
            bestell_id = shortcut.get('bestell_id') or shortcut.get('target_bestell_id')
            bestell_id_float = float(bestell_id) if bestell_id and bestell_id.isdigit() else 0.0
            
            base_record = {
                'BestellID': bestell_id_float,
                'SORTIE': shortcut.get('sortie') or shortcut.get('target_sortie') or '',
                'Bildnummer': shortcut.get('image_number') or shortcut.get('target_image_number') or '',
                'Rows': 0.0,
                'Cols': 0.0,
                'Res_m': 0.0,
                'Area_sqm': 0.0,
                'X_min': 0.0,
                'Y_min': 0.0,
                'X_max': 0.0,
                'Y_max': 0.0,
                'Has_Georef': False,
                'Image_Name': shortcut.get('target_filename', '')
            }
            
            # Check for modified/georeferenced versions
            has_georef_version = False
            
            if 'modified_files' in shortcut:
                for modified_file in shortcut['modified_files']:
                    if modified_file.suffix.lower() in ['.tif', '.tiff']:
                        area_info = self.calculate_tiff_area_extent(modified_file)
                        
                        if area_info and area_info['is_georeferenced']:
                            has_georef_version = True
                            record = base_record.copy()
                            record.update({
                                'Rows': float(area_info['rows']),
                                'Cols': float(area_info['cols']),
                                'Res_m': float((area_info['pixel_res_x_m'] + area_info['pixel_res_y_m']) / 2),
                                'Area_sqm': float(area_info['area_m2']),
                                'X_min': float(area_info['bounds'].left),
                                'Y_min': float(area_info['bounds'].bottom),
                                'X_max': float(area_info['bounds'].right),
                                'Y_max': float(area_info['bounds'].top),
                                'Has_Georef': True,
                                'Image_Name': modified_file.name
                            })
                            self.raster_data.append(record)
            
            # Add to all_images list (including non-georeferenced)
            if has_georef_version:
                # Add all georeferenced versions
                georef_records = [r for r in self.raster_data if r['Image_Name'] in 
                                 [f.name for f in shortcut.get('modified_files', [])]]
                self.all_images.extend(georef_records)
            else:
                # No georeferenced version found
                base_record['Has_Georef'] = False
                self.all_images.append(base_record)
        
        print(f"Processed {len(self.raster_data)} georeferenced images")
        return self.raster_data, self.all_images

    def display_results_table(self):
        """Display the complete image list in a table format."""
        print("\n" + "=" * 120)
        print("COMPLETE IMAGE LIST (including missing georeferenced images)")
        print("=" * 120)
        
        # Combine all images for display (avoid duplicates)
        all_images_display = []
        seen_images = set()
        
        for record in self.all_images:
            image_key = (record['BestellID'], record['Bildnummer'], record['Image_Name'])
            if image_key not in seen_images:
                seen_images.add(image_key)
                all_images_display.append(record)
        
        # Print table header
        print(f"\n{'No.':<4} {'BestellID':<10} {'SORTIE':<10} {'Bildnummer':<12} {'Rows':<6} {'Cols':<6} "
              f"{'Res(m)':<8} {'Area_sqm':<12} {'X_min':<10} {'Y_min':<10} {'X_max':<10} {'Y_max':<10} {'Georef'}")
        print("-" * 140)
        
        # Print data rows
        for i, record in enumerate(all_images_display):
            georef_status = '✓' if record['Has_Georef'] else '✗'
            print(f"{i+1:<4} "
                  f"{record['BestellID']:<10.0f} "
                  f"{record['SORTIE']:<10} "
                  f"{record['Bildnummer']:<12} "
                  f"{record['Rows']:<6.0f} "
                  f"{record['Cols']:<6.0f} "
                  f"{record['Res_m']:<8.2f} "
                  f"{record['Area_sqm']:<12.0f} "
                  f"{record['X_min']:<10.2f} "
                  f"{record['Y_min']:<10.2f} "
                  f"{record['X_max']:<10.2f} "
                  f"{record['Y_max']:<10.2f} "
                  f"{georef_status}")
        
        return all_images_display

    def save_to_csv(self, data):
        """Save the processed data to a CSV file."""
        output_path = self.output_dir / self.output_file
        
        # Ensure output directory exists
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\nSaving results to CSV: {output_path}")
        
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['BestellID', 'SORTIE', 'Bildnummer', 'Rows', 'Cols', 'Res_m', 
                         'Area_sqm', 'X_min', 'Y_min', 'X_max', 'Y_max', 'Has_Georef', 'Image_Name']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            
            writer.writeheader()
            for record in data:
                writer.writerow(record)
        
        print(f"CSV saved successfully with {len(data)} records")
        return output_path

    def display_summary_statistics(self):
        """Display summary statistics of the processed data."""
        print("\n" + "=" * 80)
        print("SUMMARY STATISTICS")
        print("=" * 80)
        
        # Combine all unique images
        all_images_display = []
        seen_images = set()
        for record in self.all_images:
            image_key = (record['BestellID'], record['Bildnummer'], record['Image_Name'])
            if image_key not in seen_images:
                seen_images.add(image_key)
                all_images_display.append(record)
        
        georef_count = sum(1 for r in all_images_display if r['Has_Georef'])
        missing_count = len(all_images_display) - georef_count
        
        print(f"Total images processed: {len(all_images_display)}")
        print(f"Georeferenced images found: {georef_count}")
        print(f"Missing georeferenced images: {missing_count}")
        
        if georef_count > 0:
            areas = [r['Area_sqm'] for r in all_images_display if r['Has_Georef']]
            print(f"\nGeoreferenced images area statistics:")
            print(f"  Total area: {sum(areas):,.0f} m² ({sum(areas)/10000:,.2f} ha)")
            print(f"  Average area: {np.mean(areas):,.0f} m² ({np.mean(areas)/10000:,.2f} ha)")
            print(f"  Min area: {min(areas):,.0f} m²")
            print(f"  Max area: {max(areas):,.0f} m²")
            
            # Extent ranges
            georef_images = [r for r in all_images_display if r['Has_Georef']]
            if georef_images:
                minXs = [r['X_min'] for r in georef_images]
                maxXs = [r['X_max'] for r in georef_images]
                minYs = [r['Y_min'] for r in georef_images]
                maxYs = [r['Y_max'] for r in georef_images]
                
                print(f"\nGeographic extent ranges:")
                print(f"  X range: {min(minXs):.4f} to {max(maxXs):.4f}")
                print(f"  Y range: {min(minYs):.4f} to {max(maxYs):.4f}")
    
    def run(self):
        """Main execution method to run the complete processing pipeline."""
        print("\n" + "=" * 80)
        print("GEOREF IMAGE PROCESSOR - STARTING PROCESSING")
        print("=" * 80)
        
        # Step 1: Find and process shortcuts
        self.find_and_process_shortcuts()
        
        # Step 2: Extract shortcut targets
        self.extract_shortcut_targets()
        
        # Step 3: Process georeferenced images
        self.process_georeferenced_images()
        
        # Step 4: Display results
        all_data = self.display_results_table()
        
        # Step 5: Save to CSV
        csv_path = self.save_to_csv(all_data)
        
        # Step 6: Display summary
        self.display_summary_statistics()
        
        print(f"\nProcessing completed successfully!")
        print(f"Output saved to: {csv_path}")
        
        return {
            'shortcuts_count': len(self.shortcuts),
            'processed_count': len(self.shortcuts_with_targets),
            'georef_count': len(self.raster_data),
            'all_images_count': len(all_data),
            'output_path': csv_path
        }


# Main execution
if __name__ == "__main__":
    # Configuration
    SOURCE_PATH = r"W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked"
    OUTPUT_DIR = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck"
    
    # Create processor instance and run
    processor = GeorefImageProcessor(SOURCE_PATH, OUTPUT_DIR)
    results = processor.run()

Initialized GeorefImageProcessor
Source path: W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked
Output directory: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck

GEOREF IMAGE PROCESSOR - STARTING PROCESSING

Scanning for shortcut files in W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked...
Found 28 shortcut files

Extracting shortcut targets and finding modified versions...


Win32 exception occurred releasing IUnknown at 0x0000016BFC5E7780
Win32 exception occurred releasing IUnknown at 0x0000016BFC5EBDA0


Successfully processed 28 shortcuts

Processing georeferenced TIFF files...
Processed 28 georeferenced images

COMPLETE IMAGE LIST (including missing georeferenced images)

No.  BestellID  SORTIE     Bildnummer   Rows   Cols   Res(m)   Area_sqm     X_min      Y_min      X_max      Y_max      Georef
--------------------------------------------------------------------------------------------------------------------------------------------
1    1737       2_3674     3242         13532  13032  0.34     20598982     556254.30  5480046.46 560708.27  5484671.32 ✓
2    1737       2_3754     4087         14637  13956  0.34     24148968     555855.81  5480113.71 560654.29  5485146.34 ✓
3    1737       2_3674     3243         13230  12881  0.34     20279284     555783.27  5479725.05 560226.73  5484288.91 ✓
4    1737       2_3754     4088         13566  13332  0.35     22043407     555403.90  5479968.11 560058.27  5484704.17 ✓
5    1737       US7_0040_A 3010         12662  12671  0.29     13786879

### Quality control through GCP file 

In [2]:
import os
import os
import sys
import numpy as np
from osgeo import gdal
from pathlib import Path
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# Windows shortcut support
try:
    import win32com.client
    HAS_WIN32COM = True
except ImportError:
    HAS_WIN32COM = False
    print("⚠️ win32com not available – .lnk resolution disabled")


# ============================================================================
# VALIDATOR CLASS
# ============================================================================

class GeoreferencingValidator:

    def __init__(self, root_folder):
        self.root_folder = Path(root_folder)

        if not self.root_folder.exists():
            raise FileNotFoundError(f"Folder does not exist: {self.root_folder}")

        self.all_results = []
        self.image_summary_stats = []
        self.shortcut_map = []
        self.current_jobs = []
        
        # NEW: Track special cases
        self.images_with_no_points = []
        self.images_with_zero_residuals = []
        self.images_with_nan_residuals = []

        # Configurable thresholds
        self.warning_threshold = 10  # px - flag for inspection
        self.critical_threshold = 20  # px - definitely needs inspection
        self.min_gcps_for_confidence = 4  # Minimum GCPs for reliable georeferencing
        
        # NEW: Threshold for "essentially zero" residuals
        self.zero_threshold = 1e-10  # Very small threshold for scientific notation zeros

    # ----------------------------------------------------------------------
    # SHORTCUT RESOLUTION
    # ----------------------------------------------------------------------

    def resolve_shortcut(self, lnk_path):
        if not HAS_WIN32COM:
            return None

        try:
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target = Path(shortcut.Targetpath)
            return target
        except Exception as e:
            print(f"❌ Error resolving shortcut {lnk_path.name}: {e}")
            return None

    # ----------------------------------------------------------------------
    # FIND SHORTCUTS → IMAGE MAP
    # ----------------------------------------------------------------------

    def find_shortcutted_images(self):
        """
        Returns a list of dicts:
        {
            shortcut_name,
            image_name,
            target_folder
        }
        """
        image_jobs = []

        for lnk in self.root_folder.rglob("*.lnk"):
            print(f"\nProcessing shortcut: {lnk.name}")
            
            target = self.resolve_shortcut(lnk)
            if not target:
                print(f"  ❌ Could not resolve shortcut")
                continue
                
            if not target.exists():
                print(f"  ❌ Target does not exist: {target}")
                continue

            # Get the shortcut name without .lnk extension
            shortcut_name = lnk.stem
            
            # Determine the clean image name based on the target
            if target.is_file() and target.suffix.lower() in ['.tif', '.tiff']:
                # Shortcut points directly to a TIFF file
                clean_image_name = target.stem
                print(f"  ✓ Shortcut points to TIFF: {target.name}")
            else:
                # Shortcut points to a folder or other file type
                # Use the shortcut name and clean it
                clean_image_name = shortcut_name
                
                # Remove common "Verknüpfung" patterns if they exist
                verknupfung_patterns = [
                    " - Verknüpfung_Finalized",
                    " - Verknüpfung",
                    " - Verknüpfung_Finalized.tif",
                    " - Verknüpfung.tif",
                    "_Verknüpfung_Finalized",
                    "_Verknüpfung"
                ]
                
                for pattern in verknupfung_patterns:
                    if pattern in clean_image_name:
                        clean_image_name = clean_image_name.replace(pattern, "")
                        print(f"  ✓ Removed pattern: {pattern}")
                
                # Also remove .tif/.tiff extensions if they're at the end
                if clean_image_name.lower().endswith('.tif'):
                    clean_image_name = clean_image_name[:-4]
                elif clean_image_name.lower().endswith('.tiff'):
                    clean_image_name = clean_image_name[:-5]
                
                clean_image_name = clean_image_name.strip()
            
            print(f"  Cleaned name: '{clean_image_name}'")
            print(f"  Target: {target}")

            image_jobs.append({
                "shortcut_name": lnk.name,
                "image_name": clean_image_name,  # Use cleaned name for .points file search
                "original_shortcut_name": shortcut_name,  # Keep original for reference
                "target_folder": target.parent if target.is_file() else target
            })

        if not image_jobs:
            print("❌ No valid shortcuts found")
        else:
            print(f"\n✓ Found {len(image_jobs)} valid shortcut-referenced images")

        return image_jobs

    # ----------------------------------------------------------------------
    # PROCESS ONE IMAGE ONLY
    # ----------------------------------------------------------------------

    def process_single_image(self, job):
        image_name = job["image_name"]
        original_name = job["original_shortcut_name"]
        folder = job["target_folder"]

        print("\n" + "="*70)
        print(f"PROCESSING IMAGE: {image_name}")
        print(f"Original shortcut: {original_name}")
        print(f"Target folder: {folder}")
        print("="*70)

        # Look for .points files (where residuals are stored)
        points_files = []
        
        # Pattern 1: ImageName.tif.points (most common)
        points_pattern1 = folder / f"{image_name}.tif.points"
        if points_pattern1.exists():
            print(f"  ✓ Found points file: {points_pattern1.name}")
            points_files.append(points_pattern1)
        
        # Pattern 2: ImageName.points (without .tif)
        points_pattern2 = folder / f"{image_name}.points"
        if points_pattern2.exists():
            print(f"  ✓ Found points file: {points_pattern2.name}")
            points_files.append(points_pattern2)
            
        # Pattern 3: ImageName_modified.tif.points
        points_pattern3 = folder / f"{image_name}_modified.tif.points"
        if points_pattern3.exists():
            print(f"  ✓ Found points file: {points_pattern3.name}")
            points_files.append(points_pattern3)
            
        # Pattern 4: Try with original name (without "Verknüpfung" patterns)
        original_cleaned = original_name
        verknupfung_patterns = [" - Verknüpfung_Finalized", " - Verknüpfung"]
        for pattern in verknupfung_patterns:
            if pattern in original_cleaned:
                original_cleaned = original_cleaned.replace(pattern, "").strip()
        
        points_pattern4 = folder / f"{original_cleaned}.tif.points"
        if points_pattern4.exists() and points_pattern4 not in points_files:
            print(f"  ✓ Found points file (from original name): {points_pattern4.name}")
            points_files.append(points_pattern4)
        
        # Pattern 5: Try with .points extension on original cleaned name
        points_pattern5 = folder / f"{original_cleaned}.points"
        if points_pattern5.exists() and points_pattern5 not in points_files:
            print(f"  ✓ Found points file (original name without .tif): {points_pattern5.name}")
            points_files.append(points_pattern5)
            
        # Pattern 6: Search for any .points file containing the image name
        if not points_files:
            print(f"\nSearching for any .points files containing '{image_name}'...")
            for points_file in folder.glob("*.points"):
                if image_name in points_file.name:
                    print(f"  ⚠️  Found (partial match): {points_file.name}")
                    points_files.append(points_file)
        
        # Also check for files in the alignment subfolder
        alignment_folder = folder / "alignment"
        if alignment_folder.exists():
            print(f"\nChecking alignment folder: {alignment_folder}")
            for points_file in alignment_folder.glob("*.points"):
                # Check for matches with both cleaned and original names
                if (image_name in points_file.name or 
                    original_cleaned in points_file.name or 
                    original_name in points_file.name):
                    print(f"  ⚠️  Found in alignment folder: {points_file.name}")
                    points_files.append(points_file)
        
        print(f"\nTotal points files found: {len(points_files)}")
        if points_files:
            for i, points_file in enumerate(points_files, 1):
                print(f"  {i}. {points_file.name}")
        else:
            print(f"❌ NO POINTS FILES FOUND FOR {image_name}")
            # NEW: Track images with no points data
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "No .points file found"
            })
            return

        image_results_map = {}
        any_points_file_processed = False

        for points_file in points_files:
            print(f"\nProcessing points file: {points_file.name}")
            results = self.process_points_file(points_file, image_name)
            if results:
                any_points_file_processed = True
                key = image_name
                image_results_map.setdefault(key, []).extend(results)
                self.all_results.extend(results)
            else:
                print(f"  ⚠️  No GCP data found in this points file")

        if not any_points_file_processed:
            print(f"❌ No GCP data found in any points file for {image_name}")
            # NEW: Track images with points files but no valid GCP data
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "Points file exists but contains no GCP data"
            })
            return

        if image_results_map:
            summaries = self.calculate_image_summary_for_folder(folder.name, image_results_map)
            self.image_summary_stats.extend(summaries)
            
            # NEW: Check if this image has zero residuals
            for summary in summaries:
                if summary.get("All_Zero_Residuals", False):
                    self.images_with_zero_residuals.append({
                        "Image": image_name,
                        "Original_Name": original_name,
                        "Folder": folder,
                        "Max_Residual": summary["Max_Residual_px"],
                        "Avg_Residual": summary["Avg_Residual_px"]
                    })
        else:
            print(f"❌ No valid GCP data found for {image_name}")
            self.images_with_no_points.append({
                "Image": image_name,
                "Original_Name": original_name,
                "Folder": folder,
                "Reason": "Points files found but no valid GCP data extracted"
            })

    # ----------------------------------------------------------------------
    # PROCESS POINTS FILE (where residuals are actually stored)
    # ----------------------------------------------------------------------

    def process_points_file(self, points_path, image_name):
        results = []

        print(f"\nProcessing points file: {points_path.name}")
        
        try:
            # Read the points file with latin-1 encoding (based on your output)
            with open(points_path, 'r', encoding='latin-1') as f:
                content = f.read()
            
            # Split into lines
            lines = content.strip().split('\n')
            
            # Find the data section (skip header lines)
            data_start = 0
            header_line = ""
            for i, line in enumerate(lines):
                if line.startswith('mapX,mapY,sourceX,sourceY,enable,dX,dY,residual'):
                    header_line = line
                    data_start = i + 1
                    break
            
            if data_start == 0:
                # Try alternative header
                for i, line in enumerate(lines):
                    if line.startswith('mapX,mapY,pixelX,pixelY,enable,dX,dY,residual'):
                        header_line = line
                        data_start = i + 1
                        break
            
            if data_start == 0:
                print(f"  ⚠️  Could not find data header in points file")
                print(f"  First few lines of the file:")
                for i, line in enumerate(lines[:5]):
                    print(f"    Line {i}: {line}")
                return results
            
            print(f"  Found header: {header_line}")
            print(f"  Found {len(lines) - data_start} GCP entries")
            
            # Track if all residuals are essentially zero
            all_essentially_zero_residuals = True
            
            # Process each GCP entry
            for i, line in enumerate(lines[data_start:], 1):
                if not line.strip():
                    continue
                    
                try:
                    # Split the line by commas
                    parts = line.strip().split(',')
                    
                    if len(parts) < 8:
                        print(f"  ⚠️  Line {i}: Not enough columns, skipping")
                        continue
                    
                    # Extract values - be more robust about column positions
                    try:
                        # Try to parse dX, dY, residual from expected positions
                        dX = float(parts[5])      # Residual in X (pixels)
                        dY = float(parts[6])      # Residual in Y (pixels)
                        residual = float(parts[7])  # Total residual (pixels)
                    except (ValueError, IndexError):
                        # If that fails, try to find columns by content
                        print(f"  ⚠️  Line {i}: Could not parse from fixed positions, trying alternative parsing")
                        print(f"     Line: {line}")
                        print(f"     Parts: {parts}")
                        
                        # Try to find float values in any column
                        for idx, part in enumerate(parts):
                            try:
                                val = float(part)
                                print(f"     Column {idx}: '{part}' -> {val}")
                            except:
                                pass
                        
                        # Skip this line if we can't parse it
                        continue
                    
                    # NEW: Check if residual is essentially zero (using scientific notation threshold)
                    if abs(residual) > self.zero_threshold:
                        all_essentially_zero_residuals = False
                    
                    # Determine individual GCP status
                    if residual < 5:
                        gcp_status = "EXCELLENT"
                    elif residual < self.warning_threshold:
                        gcp_status = "GOOD"
                    elif residual < self.critical_threshold:
                        gcp_status = "WARNING"
                    else:
                        gcp_status = "CRITICAL"
                    
                    results.append({
                        "Image_Name": image_name,
                        "GCP_ID": f"GCP_{i:03d}",
                        "Residual_X_px": dX,
                        "Residual_Y_px": dY,
                        "Total_Residual_px": residual,
                        "GCP_Status": gcp_status,
                        "Points_File": points_path.name
                    })
                    
                except ValueError as e:
                    print(f"  ⚠️  Line {i}: Error parsing line - {e}")
                    print(f"     Line content: {line}")
                    print(f"     Parts: {parts}")
                    continue
            
            print(f"  ✓ Successfully processed {len(results)} GCPs")
            
            # If all residuals are essentially zero, show more debug info
            if all_essentially_zero_residuals and len(results) > 0:
                print(f"  ⚠️  WARNING: All {len(results)} GCPs have essentially zero residuals (<{self.zero_threshold})")
                print(f"  Sample residual values:")
                for i, result in enumerate(results[:3]):
                    print(f"    GCP_{i+1}: {result['Total_Residual_px']}")
                if len(results) > 3:
                    print(f"    ... and {len(results)-3} more")
                
        except Exception as e:
            print(f"  ❌ Error processing points file: {e}")
            import traceback
            traceback.print_exc()
            
        return results

    # ----------------------------------------------------------------------
    # IMAGE CLASSIFICATION
    # ----------------------------------------------------------------------

    def calculate_image_summary_for_folder(self, folder_name, image_results_map):
        summaries = []

        for image, results in image_results_map.items():
            residuals = [r["Total_Residual_px"] for r in results]
            gcp_statuses = [r["GCP_Status"] for r in results]
            
            # Check if all residuals are essentially zero
            all_essentially_zero = all(abs(r) < self.zero_threshold for r in residuals)
            
            # Count GCPs by status
            excellent_count = gcp_statuses.count("EXCELLENT")
            good_count = gcp_statuses.count("GOOD")
            warning_count = gcp_statuses.count("WARNING")
            critical_count = gcp_statuses.count("CRITICAL")
            
            # Calculate statistics
            max_residual = max(residuals) if residuals else 0
            avg_residual = np.mean(residuals) if residuals else 0
            rmse = np.sqrt(np.mean(np.square(residuals))) if residuals else 0
            
            # Determine overall image status
            status = self.classify_image_quality(
                total_gcps=len(results),
                critical_count=critical_count,
                warning_count=warning_count,
                max_residual=max_residual,
                avg_residual=avg_residual,
                rmse=rmse,
                all_essentially_zero=all_essentially_zero
            )
            
            # Determine inspection priority
            priority = self.determine_inspection_priority(status, max_residual, critical_count, all_essentially_zero)
            
            summaries.append({
                "Image": image,
                "Status": status,
                "Priority": priority,
                "Total_GCPs": len(results),
                "Critical_GCPs": critical_count,
                "Warning_GCPs": warning_count,
                "Max_Residual_px": round(max_residual, 12),  # Show more decimal places for small values
                "Avg_Residual_px": round(avg_residual, 12),
                "RMSE_px": round(rmse, 12),
                "All_Zero_Residuals": all_essentially_zero,
                "Residual_Sample": residuals[:3] if residuals else []  # Sample of residual values
            })

            # Print summary with special warning for essentially zero residuals
            if all_essentially_zero and len(results) > 0:
                print(f"\n⚠️  {image}: {status} ({priority} priority) - ALL RESIDUALS ARE ESSENTIALLY ZERO!")
                print(f"  ⚠️  Residual values: {residuals[:3]}...")
                print(f"  ⚠️  Max: {max_residual:.12f}px, Avg: {avg_residual:.12f}px, RMSE: {rmse:.12f}px")
                print(f"  GCPs: {len(results)} total")
            else:
                print(f"\n📊 {image}: {status} ({priority} priority)")
                print(f"  Max: {max_residual:.2f}px, Avg: {avg_residual:.2f}px, RMSE: {rmse:.2f}px")
                print(f"  GCPs: {len(results)} total, {critical_count} critical, {warning_count} warnings")

        return summaries

    def classify_image_quality(self, total_gcps, critical_count, warning_count, 
                              max_residual, avg_residual, rmse, all_essentially_zero):
        """
        Simple classification based on residuals
        """
        # Rule 0: All residuals are essentially zero
        if all_essentially_zero:
            return "ZERO_RESIDUALS"
        
        # Rule 1: Not enough GCPs
        if total_gcps < self.min_gcps_for_confidence:
            return "INSUFFICIENT_GCPs"
        
        # Rule 2: Any critical GCPs
        if critical_count > 0:
            return "FAIL"
        
        # Rule 3: Too many warnings (>50% of GCPs)
        if warning_count > 0 and warning_count / total_gcps > 0.5:
            return "NEEDS_INSPECTION"
        
        # Rule 4: High RMSE indicates systematic error
        if rmse > 15:
            return "FAIL"
        
        # Rule 5: High max residual
        if max_residual > 15:
            return "NEEDS_INSPECTION"
        
        # Rule 6: Moderate issues
        if warning_count > 0 or avg_residual > 8:
            return "NEEDS_INSPECTION"
        
        # Default: Pass
        return "PASS"

    def determine_inspection_priority(self, status, max_residual, critical_count, all_essentially_zero):
        """
        Determine inspection priority
        """
        if status == "ZERO_RESIDUALS":
            return "VERY_LOW"  # These need special attention but not for quality issues
        
        if status == "PASS":
            return "LOW"
        elif status == "INSUFFICIENT_GCPs":
            return "MEDIUM"
        elif critical_count > 0:
            if max_residual > 30:
                return "CRITICAL"
            else:
                return "HIGH"
        elif status == "NEEDS_INSPECTION":
            return "MEDIUM"
        else:  # FAIL
            return "HIGH"

    # ----------------------------------------------------------------------
    # DISPLAY RESULTS IN CONSOLE
    # ----------------------------------------------------------------------

    def display_results(self):
        print("\n" + "="*80)
        print("GEOREFERENCING VALIDATION RESULTS")
        print("="*80)
        
        # NEW: Display images with no points data
        if self.images_with_no_points:
            print(f"\n🚫 IMAGES WITH NO POINTS DATA ({len(self.images_with_no_points)} images)")
            print("-"*60)
            for img in self.images_with_no_points:
                print(f"• {img['Image']}")
                print(f"  Original: {img['Original_Name']}")
                print(f"  Folder: {img['Folder']}")
                print(f"  Reason: {img['Reason']}")
                print()
        
        # NEW: Display images with zero residuals
        if self.images_with_zero_residuals:
            print(f"\n🔍 IMAGES WITH ESSENTIALLY ZERO RESIDUALS ({len(self.images_with_zero_residuals)} images)")
            print("-"*60)
            print("These images have residuals < 1e-10 (scientific notation zeros)")
            print("This usually indicates:")
            print("  1. Already georeferenced/rectified images")
            print("  2. Perfect mathematical fit (unlikely in practice)")
            print("  3. Points file format differences")
            print()
            for img in self.images_with_zero_residuals:
                print(f"• {img['Image']}")
                print(f"  Original: {img['Original_Name']}")
                print(f"  Folder: {img['Folder']}")
                print(f"  Max Residual: {img['Max_Residual']:.12f} px")
                print()
        
        # Display GCP details if available
        if self.all_results:
            print("\n📋 DETAILED GCP ANALYSIS")
            print("-"*40)
            
            # Group results by image
            image_groups = {}
            for result in self.all_results:
                image_name = result["Image_Name"]
                if image_name not in image_groups:
                    image_groups[image_name] = []
                image_groups[image_name].append(result)
            
            # Display each image's GCPs
            for image_name, gcp_results in image_groups.items():
                print(f"\n📷 {image_name}:")
                print(f"{'-'*30}")
                
                # Check if all residuals are essentially zero for this image
                all_essentially_zero = all(abs(g["Total_Residual_px"]) < self.zero_threshold for g in gcp_results)
                if all_essentially_zero and len(gcp_results) > 0:
                    print(f"⚠️  ALL RESIDUALS ARE ESSENTIALLY ZERO (<{self.zero_threshold})")
                
                # Display GCP table
                print(f"{'GCP':<8} {'Residual (px)':<25} {'Status':<15}")
                print(f"{'-'*55}")
                
                for gcp in gcp_results:
                    gcp_id = gcp["GCP_ID"]
                    residual = gcp["Total_Residual_px"]
                    status = gcp["GCP_Status"]
                    
                    # Format residual with scientific notation for very small values
                    if abs(residual) < 0.001:
                        residual_display = f"{residual:.12e}"
                    else:
                        residual_display = f"{residual:.4f}"
                    
                    # Add status indicators
                    if status == "CRITICAL":
                        status_display = f"🔴 {status}"
                    elif status == "WARNING":
                        status_display = f"🟡 {status}"
                    elif status == "GOOD":
                        status_display = f"🟢 {status}"
                    else:
                        status_display = f"✅ {status}"
                    
                    # Highlight essentially zero residuals
                    if abs(residual) < self.zero_threshold:
                        residual_display = f"{residual_display:<25} ⚠️"
                    else:
                        residual_display = f"{residual_display:<25}"
                    
                    print(f"{gcp_id:<8} {residual_display} {status_display:<15}")
        
        # Display image summary if available
        if self.image_summary_stats:
            print("\n\n📊 IMAGE VALIDATION SUMMARY")
            print("="*80)
            
            # Display summary table
            print(f"\n{'Image':<40} {'Status':<25} {'Priority':<15} {'Max Res (px)':<20} {'GCPs':<8}")
            print(f"{'-'*115}")
            
            for summary in self.image_summary_stats:
                image = summary["Image"]
                status = summary["Status"]
                priority = summary["Priority"]
                max_res = summary["Max_Residual_px"]
                total_gcps = summary["Total_GCPs"]
                all_essentially_zero = summary.get("All_Zero_Residuals", False)
                
                # Add priority indicators
                if priority == "CRITICAL":
                    priority_display = f"🔴 {priority}"
                elif priority == "HIGH":
                    priority_display = f"🟠 {priority}"
                elif priority == "MEDIUM":
                    priority_display = f"🟡 {priority}"
                elif priority == "VERY_LOW":
                    priority_display = f"⚪ {priority}"
                else:
                    priority_display = f"🟢 {priority}"
                
                # Add status indicators
                if status == "ZERO_RESIDUALS":
                    status_display = f"🔵 ZERO_RESIDUALS"
                elif status == "FAIL":
                    status_display = f"🔴 {status}"
                elif status == "NEEDS_INSPECTION":
                    status_display = f"🟡 {status}"
                elif status == "INSUFFICIENT_GCPs":
                    status_display = f"⚫ {status}"
                else:
                    status_display = f"✅ {status}"
                
                # Format max residual for display
                if abs(max_res) < 0.001:
                    max_res_display = f"{max_res:.12e}"
                else:
                    max_res_display = f"{max_res:.4f}"
                
                # Highlight zero residuals
                if all_essentially_zero:
                    max_res_display = f"{max_res_display:<20} ⚠️"
                else:
                    max_res_display = f"{max_res_display:<20}"
                
                print(f"{image:<40} {status_display:<25} {priority_display:<15} {max_res_display} {total_gcps:<8}")
            
            # Display overall statistics
            self.display_overall_statistics()
        
        # Display classification guidelines
        self.display_classification_guidelines()
        
        print("\n" + "="*80)
        print("VALIDATION COMPLETE")
        print("="*80)
        
        # NEW: Final summary of special cases
        print("\n📈 FINAL SUMMARY")
        print("-"*40)
        print(f"Total images processed: {len(self.current_jobs)}")
        print(f"Images with valid GCP data: {len(self.image_summary_stats)}")
        print(f"Images with no points data: {len(self.images_with_no_points)}")
        print(f"Images with essentially zero residuals: {len(self.images_with_zero_residuals)}")
        
        if self.images_with_no_points:
            print(f"\n⚠️  ACTION NEEDED: Check these {len(self.images_with_no_points)} images for missing .points files")
        
        if self.images_with_zero_residuals:
            print(f"\n🔍 INSPECT: {len(self.images_with_zero_residuals)} images have essentially zero residuals")
            print("   These may already be georeferenced or need manual verification")

    def display_overall_statistics(self):
        """Display overall statistics about the validation"""
        if not self.image_summary_stats:
            return
        
        print(f"\n\n📈 OVERALL STATISTICS")
        print("-"*40)
        
        # Count images by status
        status_counts = {}
        priority_counts = {}
        zero_residual_count = 0
        
        for summary in self.image_summary_stats:
            status = summary["Status"]
            priority = summary["Priority"]
            all_zero = summary.get("All_Zero_Residuals", False)
            
            status_counts[status] = status_counts.get(status, 0) + 1
            priority_counts[priority] = priority_counts.get(priority, 0) + 1
            if all_zero:
                zero_residual_count += 1
        
        # Calculate percentages
        total_images = len(self.image_summary_stats)
        
        print(f"\n📁 Images with GCP data: {total_images}")
        print(f"📊 Total GCPs Analyzed: {len(self.all_results)}")
        print(f"🔍 Images with essentially zero residuals: {zero_residual_count}")
        
        if zero_residual_count > 0:
            print(f"   ⚠️  These images have residuals < {self.zero_threshold}")
            print(f"   Possible causes: already georeferenced, perfect fit, or format issue")
        
        print(f"\n📊 STATUS DISTRIBUTION:")
        for status, count in sorted(status_counts.items()):
            percentage = (count / total_images) * 100
            print(f"  {status:<25} : {count:>3} images ({percentage:>5.1f}%)")
        
        print(f"\n🎯 PRIORITY DISTRIBUTION:")
        priority_order = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "VERY_LOW"]
        for priority in priority_order:
            if priority in priority_counts:
                count = priority_counts[priority]
                percentage = (count / total_images) * 100
                print(f"  {priority:<25} : {count:>3} images ({percentage:>5.1f}%)")
        
        # Calculate average metrics (excluding images with all essentially zero residuals)
        non_zero_images = [s for s in self.image_summary_stats if not s.get("All_Zero_Residuals", False)]
        
        if non_zero_images:
            avg_max_residual = np.mean([s["Max_Residual_px"] for s in non_zero_images])
            avg_rmse = np.mean([s["RMSE_px"] for s in non_zero_images])
            
            print(f"\n📐 AVERAGE METRICS (excluding {zero_residual_count} images with essentially zero residuals):")
            print(f"  Average Max Residual: {avg_max_residual:.4f} px")
            print(f"  Average RMSE: {avg_rmse:.4f} px")
        else:
            print(f"\n📐 AVERAGE METRICS: All images have essentially zero residuals")

    def display_classification_guidelines(self):
        """Display the quality classification and priority assignment guidelines"""
        print("\n\n📚 QUALITY CLASSIFICATION GUIDELINES")
        print("="*80)
        
        print("\n🎯 QUALITY CLASSIFICATION")
        print("-"*30)
        print(f"• 🔵 ZERO_RESIDUALS: All residuals < {self.zero_threshold}")
        print("• ✅ PASS: Good quality (<10px max residual)")
        print("• ⚠️  NEEDS_INSPECTION: Moderate issues (10-20px residuals)")
        print("• 🔴 FAIL: Critical issues (>20px residuals)")
        print("• ⚫ INSUFFICIENT_GCPs: Too few control points (<4 GCPs)")
        
        print("\n📊 GCP CLASSIFICATION")
        print("-"*30)
        print("• ✅ EXCELLENT: <5px residual")
        print("• 🟢 GOOD: 5-10px residual")
        print("• 🟡 WARNING: 10-20px residual")
        print("• 🔴 CRITICAL: >20px residual")
        
        print("\n🎯 PRIORITY ASSIGNMENT")
        print("-"*30)
        print("• 🔴 CRITICAL: Max residual >30px or multiple critical GCPs")
        print("• 🟠 HIGH: Critical GCPs present")
        print("• 🟡 MEDIUM: Needs inspection but less urgent")
        print("• 🟢 LOW: Minor issues or passes")
        print("• ⚪ VERY_LOW: Essentially zero residuals (already georeferenced?)")
        
        print("\n📝 INTERPRETATION:")
        print("-"*30)
        print("• Images with 🔴 CRITICAL or 🟠 HIGH priority need immediate attention")
        print("• Images with 🟡 MEDIUM priority should be reviewed when possible")
        print("• Images with 🟢 LOW priority are acceptable for most applications")
        print(f"• Images with ⚪ VERY_LOW priority have residuals < {self.zero_threshold}")
        
        print("\n⚠️  SPECIAL NOTES:")
        print("-"*30)
        print(f"• Images with ALL residuals < {self.zero_threshold} may already be georeferenced")
        print("• Check if these images need manual verification")
        print("• Zero residuals can indicate perfect mathematical fit (rare in practice)")

    # ----------------------------------------------------------------------
    # MAIN RUN
    # ----------------------------------------------------------------------

    def run_validation(self):
        print("GEOREFERENCING VALIDATION")
        print("="*60)
        print(f"Root folder: {self.root_folder}")
        print("="*60)
        
        jobs = self.find_shortcutted_images()
        self.current_jobs = jobs  # Store for reference
        
        if not jobs:
            print("❌ No valid shortcut jobs to process")
            return
            
        print(f"\nStarting validation of {len(jobs)} images...")
        
        for i, job in enumerate(jobs, 1):
            print(f"\n{'='*60}")
            print(f"Progress: {i}/{len(jobs)}")
            self.process_single_image(job)

        print(f"\n{'='*60}")
        print("VALIDATION PROCESSING COMPLETE")
        print("="*60)
        
        # Display results instead of generating CSV files
        self.display_results()


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Update this path to your actual folder
    MAIN_FOLDER = r"W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked"

    try:
        validator = GeoreferencingValidator(MAIN_FOLDER)
        validator.run_validation()
    except FileNotFoundError as e:
        print(f"❌ ERROR: {e}")
        print("Please check that the folder path is correct.")
    except Exception as e:
        print(f"❌ UNEXPECTED ERROR: {e}")
        import traceback
        traceback.print_exc()

    print("\n" + "="*60)
    print("SCRIPT FINISHED")
    print("="*60)
    input("Press Enter to exit...")

GEOREFERENCING VALIDATION
Root folder: W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked

Processing shortcut: 3242#_#2_3674#ID1737#23_07_1947 - Verknüpfung.lnk
  ✓ Shortcut points to TIFF: 3242#_#2_3674#ID1737#23_07_1947.tif
  Cleaned name: '3242#_#2_3674#ID1737#23_07_1947'
  Target: W:\1947_07_23#_#2_3674\3242#_#2_3674#ID1737#23_07_1947.tif

Processing shortcut: 4087#_#2_3754#ID1737#13_09_1947 - Verknüpfung.lnk
  ✓ Shortcut points to TIFF: 4087#_#2_3754#ID1737#13_09_1947.tif
  Cleaned name: '4087#_#2_3754#ID1737#13_09_1947'
  Target: W:\1947_09_13#_#2_3754\4087#_#2_3754#ID1737#13_09_1947.tif

Processing shortcut: 3243#_#2_3674#ID1737#23_07_1947.tif.lnk
  ✓ Shortcut points to TIFF: 3243#_#2_3674#ID1737#23_07_1947.tif
  Cleaned name: '3243#_#2_3674#ID1737#23_07_1947'
  Target: W:\1947_07_23#_#2_3674\3243#_#2_3674#ID1737#23_07_1947.tif

Processing shortcut: 4088#_#2_3754#ID1737#13_09_1947.tif.lnk
  ✓ Shortcut points to TIFF: 4088#_#2_3754#ID1737#13_09

## DATABASE

### Check for Frame Areas, Bewölkung Info (Requires Input Path)

In [5]:
import os
import re
from pathlib import Path
import psycopg2
import pandas as pd

class PostGISImageProcessor:
    """Class to process PostGIS images from .lnk files and query database"""
    
    def __init__(self, base_path, output_path, db_config):
        """
        Initialize the processor
        
        Args:
            base_path (str): Path to folder containing .lnk files
            output_path (str): Path to save output CSV
            db_config (dict): Database connection parameters
        """
        self.base_path = Path(base_path)
        self.output_path = Path(output_path)
        self.db_config = db_config
        self.lnk_files = []
        self.image_data = []
        self.results_df = None
        
    def get_lnk_files(self):
        """Find all .lnk files in the base path and subfolders"""
        self.lnk_files = []
        for root, _, files in os.walk(self.base_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    self.lnk_files.append(Path(root) / file)
        return len(self.lnk_files)
    
    @staticmethod
    def parse_image_filename(filename):
        """
        Parse filename to extract ImageNumber, BestellID, SORTIE
        
        Args:
            filename (str): Filename to parse
            
        Returns:
            dict: Parsed image data or None if parsing fails
        """
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').strip()
        parts = filename.split('#')
        
        if len(parts) >= 4:
            image_number = parts[0].strip()
            
            bestell_id = None
            for part in parts:
                id_match = re.search(r'ID(\d+)', part, re.IGNORECASE)
                if id_match:
                    bestell_id = int(id_match.group(1))
                    break
            
            sortie = parts[2].strip() if len(parts) > 2 else None
            
            return {
                'ImageNumber': image_number,
                'BestellID': bestell_id,
                'SORTIE': sortie
            }
        return None
    
    def parse_all_filenames(self):
        """Parse all .lnk filenames"""
        self.image_data = []
        for lnk_file in self.lnk_files:
            parsed = self.parse_image_filename(lnk_file.name)
            if parsed:
                self.image_data.append(parsed)
        return len(self.image_data)
    
    @staticmethod
    def extract_numbers_from_string(value):
        """
        Extract all integer numbers from a string that may contain separators
        
        Args:
            value (str): String value that may contain numbers with separators
            
        Returns:
            list: List of extracted integers
        """
        if not value or pd.isna(value):
            return []
        
        # Find all sequences of digits
        numbers = re.findall(r'\d+', str(value))
        return [int(num) for num in numbers]
    
    def query_database(self):
        """Query database for images and store results"""
        if not self.image_data:
            self.results_df = pd.DataFrame()
            return self.results_df
        
        # First, check what fields are available in the table
        table_columns = self.get_table_columns()
        
        # Check if 'weitere Bestell ID' field exists
        has_weitere_bestell_id = any('weitere bestell' in col.lower() for col in table_columns)
        
        # Check if 'Qualität Bild_Studis' field exists
        has_qualitaet_bild_studis = 'Qualität Bild_Studis' in table_columns
        
        conditions, params = [], []
        for img in self.image_data:
            if img.get('BestellID') and img.get('SORTIE') and img.get('ImageNumber'):
                # Base condition for exact match with 'Bestell ID'
                condition = '("Bestell ID" = %s AND sortie_ori = %s AND "Bildnummer" = %s)'
                
                # If 'weitere Bestell ID' field exists, also check it
                if has_weitere_bestell_id:
                    condition = f'({condition} OR ("weitere Bestell ID" LIKE %s AND sortie_ori = %s AND "Bildnummer" = %s))'
                    # For 'weitere Bestell ID', we need to check if our BestellID is in the string
                    params.extend([
                        img['BestellID'], img['SORTIE'], img['ImageNumber'],  # For exact 'Bestell ID' match
                        f'%{img["BestellID"]}%', img['SORTIE'], img['ImageNumber']  # For 'weitere Bestell ID' match
                    ])
                else:
                    params.extend([img['BestellID'], img['SORTIE'], img['ImageNumber']])
                
                conditions.append(condition)
        
        if not conditions:
            self.results_df = pd.DataFrame()
            return self.results_df
        
        # Build SELECT clause with additional fields
        select_clause = """
        SELECT 
            "Bestell ID", sortie_ori, "Bildnummer", "Bewölkung"
        """
        
        if has_weitere_bestell_id:
            select_clause += ', "weitere Bestell ID"'
        
        if has_qualitaet_bild_studis:
            select_clause += ', "Qualität Bild_Studis"'
        
        select_clause += """,
            ROUND(ST_Area(geom)::numeric, 6) AS area_sqm,
            ROUND((ST_XMax(geom) - ST_XMin(geom))::numeric, 6) AS width_m,
            ROUND((ST_YMax(geom) - ST_YMin(geom))::numeric, 6) AS height_m,
            ROUND(ST_XMin(geom)::numeric, 6) AS xmin,
            ROUND(ST_YMin(geom)::numeric, 6) AS ymin,
            ROUND(ST_XMax(geom)::numeric, 6) AS xmax,
            ROUND(ST_YMax(geom)::numeric, 6) AS ymax,
            ROUND(ST_X(ST_Centroid(geom))::numeric, 6) AS centroid_x,
            ROUND(ST_Y(ST_Centroid(geom))::numeric, 6) AS centroid_y
        FROM bildumrahmungen_aktuell 
        WHERE {where_clause}
        ORDER BY "Bestell ID", sortie_ori, "Bildnummer"
        """
        
        query = select_clause.format(where_clause=' OR '.join(conditions))
        
        try:
            conn = psycopg2.connect(**self.db_config)
            self.results_df = pd.read_sql_query(query, conn, params=params)
            conn.close()
            
            # If 'weitere Bestell ID' exists, extract numbers from it
            if has_weitere_bestell_id and 'weitere Bestell ID' in self.results_df.columns:
                self.results_df['weitere_Bestell_ID_numbers'] = self.results_df['weitere Bestell ID'].apply(
                    lambda x: self.extract_numbers_from_string(x)
                )
            
            return self.results_df
        except Exception as e:
            print(f"Database error: {e}")
            self.results_df = pd.DataFrame()
            return self.results_df
    
    def get_table_columns(self):
        """Get column names from the database table"""
        try:
            conn = psycopg2.connect(**self.db_config)
            cursor = conn.cursor()
            cursor.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_name = 'bildumrahmungen_aktuell'
                ORDER BY ordinal_position;
            """)
            columns = [row[0] for row in cursor.fetchall()]
            cursor.close()
            conn.close()
            return columns
        except Exception as e:
            print(f"Error getting table columns: {e}")
            return []
    
    def create_output_dataframe(self):
        """Create formatted output DataFrame from query results"""
        if self.results_df is None or self.results_df.empty:
            return pd.DataFrame()
        
        output_df = pd.DataFrame()
        
        # Add identifier columns
        output_df['BestellID'] = self.results_df['Bestell ID'].astype(int)
        output_df['SORTIE'] = self.results_df['sortie_ori'].astype(str)
        output_df['Bildnummer'] = self.results_df['Bildnummer'].astype(str)
        output_df['Bewölkung'] = self.results_df['Bewölkung'].fillna('').astype(str)
        
        # Add 'Qualität Bild_Studis' if available
        if 'Qualität Bild_Studis' in self.results_df.columns:
            output_df['Qualität_Bild_Studis'] = self.results_df['Qualität Bild_Studis'].fillna('').astype(str)
        
        # Add 'weitere Bestell ID' information if available
        if 'weitere Bestell ID' in self.results_df.columns:
            output_df['weitere_Bestell_ID_raw'] = self.results_df['weitere Bestell ID'].fillna('').astype(str)
        
        if 'weitere_Bestell_ID_numbers' in self.results_df.columns:
            output_df['weitere_Bestell_ID_numbers'] = self.results_df['weitere_Bestell_ID_numbers'].apply(
                lambda x: ', '.join(map(str, x)) if x else ''
            )
        
        # Add spatial columns as double precision
        spatial_columns = {
            'area_sqm': 'Area_sqm',
            'width_m': 'Width_m',
            'height_m': 'Height_m',
            'xmin': 'X_min',
            'ymin': 'Y_min',
            'xmax': 'X_max',
            'ymax': 'Y_max',
            'centroid_x': 'Centroid_X',
            'centroid_y': 'Centroid_Y'
        }
        
        for src_col, dest_col in spatial_columns.items():
            if src_col in self.results_df.columns:
                output_df[dest_col] = pd.to_numeric(self.results_df[src_col], errors='coerce').astype('float64')
        
        return output_df
    
    def save_to_csv(self, output_df, filename="DB_imageExtent.csv"):
        """Save DataFrame to CSV file"""
        if output_df.empty:
            return None
        
        self.output_path.mkdir(parents=True, exist_ok=True)
        output_file = self.output_path / filename
        
        output_df.to_csv(output_file, index=False, float_format='%.6f', encoding='utf-8-sig')
        
        return output_file
    
    def display_results(self, output_df, max_rows=None):
        """Display the results table"""
        if output_df.empty:
            print("No results to display")
            return
        
        print(f"\n{'='*120}")
        print(f"COMPLETE IMAGE LIST ({len(output_df)} images)")
        print(f"{'='*120}")
        
        pd.set_option('display.float_format', '{:.6f}'.format)
        pd.set_option('display.max_rows', max_rows or len(output_df))
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', 20)
        
        print(output_df.to_string(index=False))
        
        # Reset display options
        pd.reset_option('display.float_format')
        pd.reset_option('display.max_rows')
    
    def process(self, output_filename="DB_imageExtent.csv"):
        """Main processing method - executes all steps"""
        print(f"Processing .lnk files from: {self.base_path}")
        
        # Step 1: Get .lnk files
        file_count = self.get_lnk_files()
        print(f"Found {file_count} .lnk files")
        
        # Step 2: Parse filenames
        parsed_count = self.parse_all_filenames()
        print(f"Parsed {parsed_count} files")
        
        # Step 3: Check database table structure
        table_columns = self.get_table_columns()
        print(f"\nDatabase table has {len(table_columns)} columns")
        
        # Check for 'weitere Bestell ID' field
        weitere_fields = [col for col in table_columns if 'weitere' in col.lower()]
        if weitere_fields:
            print(f"Found 'weitere Bestell ID' field(s): {', '.join(weitere_fields)}")
        
        # Check for 'Qualität Bild_Studis' field
        if 'Qualität Bild_Studis' in table_columns:
            print(f"Found 'Qualität Bild_Studis' field")
        else:
            print(f"Note: 'Qualität Bild_Studis' field not found in table")
        
        # Step 4: Query database
        if self.image_data:
            self.query_database()
            
            if not self.results_df.empty:
                print(f"\nFound {len(self.results_df)} matching records in database")
                
                # Step 5: Create output DataFrame
                output_df = self.create_output_dataframe()
                
                # Step 6: Save to CSV
                output_file = self.save_to_csv(output_df, output_filename)
                print(f"Output saved to: {output_file}")
                
                # Step 7: Display results
                self.display_results(output_df)
                
                # Show summary of 'weitere Bestell ID' matches
                if 'weitere_Bestell_ID_numbers' in output_df.columns:
                    weitere_matches = output_df[
                        output_df['weitere_Bestell_ID_numbers'].str.len() > 0
                    ]
                    if len(weitere_matches) > 0:
                        print(f"\n{'='*80}")
                        print(f"MATCHES VIA 'WEITERE BESTELL ID' FIELD:")
                        print(f"{'='*80}")
                        print(f"Total images matched via 'weitere Bestell ID': {len(weitere_matches)}")
                        for _, row in weitere_matches.head(5).iterrows():
                            print(f"  BestellID: {row['BestellID']}, Bildnummer: {row['Bildnummer']}, "
                                  f"weitere IDs: {row['weitere_Bestell_ID_numbers']}")
                        if len(weitere_matches) > 5:
                            print(f"  ... and {len(weitere_matches) - 5} more")
                
                return output_df, output_file
            else:
                print("No matching records found in database")
                return None, None
        else:
            print("No valid image data to process")
            return None, None


# ========== CONFIGURATION ==========
BASE_PATH = r"W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked"
OUTPUT_PATH = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck"

DB_CONFIG = {
    'host': '192.168.190.10',
    'port': 5432,
    'database': 'Server_LBA',
    'user': 'LBA_team',
    'password': 'lba',
    'sslmode': 'prefer'
}

# ========== MAIN EXECUTION ==========
if __name__ == "__main__":
    # Create processor instance
    processor = PostGISImageProcessor(
        base_path=BASE_PATH,
        output_path=OUTPUT_PATH,
        db_config=DB_CONFIG
    )
    
    # Run the complete processing pipeline
    output_df, output_file = processor.process()
    
    if output_df is not None:
        print(f"\nProcessing completed successfully!")
        print(f"Total images processed: {len(output_df)}")
        print(f"Output file: {output_file}")

Processing .lnk files from: W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked
Found 28 .lnk files
Parsed 28 files

Database table has 48 columns
Found 'weitere Bestell ID' field(s): weitere Bestell ID, weitere Bestell ID
Found 'Qualität Bild_Studis' field

Found 28 matching records in database
Output saved to: G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DB_imageExtent.csv

COMPLETE IMAGE LIST (28 images)
 BestellID             SORTIE Bildnummer Bewölkung Qualität_Bild_Studis weitere_Bestell_ID_raw weitere_Bestell_ID_numbers         Area_sqm      Width_m     Height_m         X_min          Y_min         X_max          Y_max    Centroid_X     Centroid_Y
      1737             2_3674     3242.0       0.0                    B                                                     9434998.013627  4121.069699  4269.186638 556389.735031 5480254.380509 560510.804730 5484523.567147 558421.589276 5482390.417113
      1737      

### Check for Extent Overlap

In [6]:
import pandas as pd
import numpy as np
import os

class ImageOverlapAnalyzer:
    def __init__(self, db_path, raster_path):
        self.db_path = db_path
        self.raster_path = raster_path
        self.db_df = None
        self.raster_df = None
        self.results_df = None
        
        # Column mappings
        self.bestellid_col_db = None
        self.sortie_col_db = None
        self.bildnummer_col_db = None
        self.bestellid_col_raster = None
        self.sortie_col_raster = None
        self.bildnummer_col_raster = None
        self.db_extent_cols = {}
        self.raster_extent_cols = {}
    
    def load_data(self):
        """Load and preprocess CSV files"""
        self.db_df = pd.read_csv(self.db_path)
        self.raster_df = pd.read_csv(self.raster_path)
        
        # Clean column names
        self.db_df = self._clean_column_names(self.db_df)
        self.raster_df = self._clean_column_names(self.raster_df)
        
        # Find key columns
        self._identify_columns()
    
    def _clean_column_names(self, df):
        """Clean column names by removing special characters"""
        df_clean = df.copy()
        df_clean.columns = [col.strip().lower().replace('-', '').replace('_', '').replace('@', '') 
                           for col in df_clean.columns]
        return df_clean
    
    def _find_column(self, df, keywords):
        """Find column based on keywords"""
        for col in df.columns:
            clean_col = col.lower()
            for keyword in keywords:
                if keyword.lower() in clean_col:
                    return col
        return None
    
    def _identify_columns(self):
        """Identify all important columns in both dataframes"""
        # Database columns
        self.bestellid_col_db = self._find_column(self.db_df, ['bestellid', 'bestell', 'orderid'])
        self.sortie_col_db = self._find_column(self.db_df, ['sortie', 'flight', 'mission'])
        self.bildnummer_col_db = self._find_column(self.db_df, ['bildnummer', 'imagenumber', 'imageno', 'bildnr'])
        
        # Raster columns
        self.bestellid_col_raster = self._find_column(self.raster_df, ['bestellid', 'bestell', 'orderid'])
        self.sortie_col_raster = self._find_column(self.raster_df, ['sortie', 'flight', 'mission'])
        self.bildnummer_col_raster = self._find_column(self.raster_df, ['bildnummer', 'imagenumber', 'imageno', 'bildnr'])
        
        # Extent columns
        self.db_extent_cols = self._find_extent_columns(self.db_df)
        self.raster_extent_cols = self._find_extent_columns(self.raster_df)
    
    def _find_extent_columns(self, df):
        """Find extent columns (xmin, xmax, ymin, ymax)"""
        extent_cols = {}
        for col in df.columns:
            if 'xmin' in col:
                extent_cols['xmin'] = col
            elif 'xmax' in col:
                extent_cols['xmax'] = col
            elif 'ymin' in col:
                extent_cols['ymin'] = col
            elif 'ymax' in col:
                extent_cols['ymax'] = col
        return extent_cols
    
    def _calculate_overlap(self, rect1, rect2):
        """Calculate overlap percentage between two rectangles"""
        # rect format: (xmin, ymin, xmax, ymax)
        
        # Calculate intersection
        x_left = max(rect1[0], rect2[0])
        y_bottom = max(rect1[1], rect2[1])
        x_right = min(rect1[2], rect2[2])
        y_top = min(rect1[3], rect2[3])
        
        if x_right < x_left or y_top < y_bottom:
            return 0.0
        
        # Calculate areas
        area_intersection = (x_right - x_left) * (y_top - y_bottom)
        area_rect1 = (rect1[2] - rect1[0]) * (rect1[3] - rect1[1])
        area_rect2 = (rect2[2] - rect2[0]) * (rect2[3] - rect2[1])
        
        if area_rect1 == 0 or area_rect2 == 0:
            return 0.0
        
        # Return average overlap percentage
        overlap1 = (area_intersection / area_rect1) * 100
        overlap2 = (area_intersection / area_rect2) * 100
        return (overlap1 + overlap2) / 2
    
    def _find_db_match(self, raster_row):
        """Find matching database entry for a raster row using SORTIE-first matching"""
        # Get raster data
        raster_bildnummer = raster_row[self.bildnummer_col_raster] if self.bildnummer_col_raster and pd.notna(raster_row[self.bildnummer_col_raster]) else None
        raster_bestellid = raster_row[self.bestellid_col_raster] if self.bestellid_col_raster and pd.notna(raster_row[self.bestellid_col_raster]) else None
        raster_sortie = raster_row[self.sortie_col_raster] if self.sortie_col_raster and pd.notna(raster_row[self.sortie_col_raster]) else None
        
        db_match = None
        
        # FIRST: Match by SORTIE + BestellID + Bildnummer combination
        if all([self.sortie_col_db, self.bestellid_col_db, self.bildnummer_col_db,
                raster_sortie is not None, raster_bestellid is not None, raster_bildnummer is not None]):
            condition = (self.db_df[self.sortie_col_db] == raster_sortie) & \
                       (self.db_df[self.bestellid_col_db] == raster_bestellid) & \
                       (self.db_df[self.bildnummer_col_db] == raster_bildnummer)
            db_matches = self.db_df[condition]
            if not db_matches.empty:
                db_match = db_matches.iloc[0]
        
        # SECOND: If no match, try SORTIE + Bildnummer (without BestellID)
        if db_match is None and self.sortie_col_db and self.bildnummer_col_db:
            if raster_sortie is not None and raster_bildnummer is not None:
                condition = (self.db_df[self.sortie_col_db] == raster_sortie) & \
                           (self.db_df[self.bildnummer_col_db] == raster_bildnummer)
                db_matches = self.db_df[condition]
                if not db_matches.empty:
                    db_match = db_matches.iloc[0]
        
        # THIRD: If still no match, try just SORTIE
        if db_match is None and self.sortie_col_db and self.bildnummer_col_db:
            if raster_sortie is not None and raster_bildnummer is not None:
                condition = (self.db_df[self.sortie_col_db] == raster_sortie)
                db_matches = self.db_df[condition]
                # Look for matching Bildnummer within the same SORTIE
                for _, db_candidate in db_matches.iterrows():
                    if db_candidate[self.bildnummer_col_db] == raster_bildnummer:
                        db_match = db_candidate
                        break
        
        return db_match, raster_bestellid, raster_sortie, raster_bildnummer
    
    def analyze_overlap(self):
        """Main analysis function to calculate overlaps for all images"""
        if self.db_df is None or self.raster_df is None:
            self.load_data()
        
        results = []
        
        # Process each raster image
        for _, raster_row in self.raster_df.iterrows():
            # Find matching database entry
            db_match, raster_bestellid, raster_sortie, raster_bildnummer = self._find_db_match(raster_row)
            
            # Initialize result values
            db_bildnummer = None
            overlap_pct = 0.0
            
            # Calculate overlap if match found
            if db_match is not None:
                db_bildnummer = db_match[self.bildnummer_col_db] if self.bildnummer_col_db and pd.notna(db_match[self.bildnummer_col_db]) else None
                
                # Get extents
                try:
                    db_extent = (
                        float(db_match[self.db_extent_cols['xmin']]),
                        float(db_match[self.db_extent_cols['ymin']]),
                        float(db_match[self.db_extent_cols['xmax']]),
                        float(db_match[self.db_extent_cols['ymax']])
                    )
                    
                    raster_extent = (
                        float(raster_row[self.raster_extent_cols['xmin']]),
                        float(raster_row[self.raster_extent_cols['ymin']]),
                        float(raster_row[self.raster_extent_cols['xmax']]),
                        float(raster_row[self.raster_extent_cols['ymax']])
                    )
                    
                    overlap_pct = self._calculate_overlap(db_extent, raster_extent)
                except (KeyError, ValueError, TypeError, AttributeError):
                    overlap_pct = 0.0
            
            # Add to results
            results.append({
                'BestellID': raster_bestellid,
                'SORTIE': raster_sortie,
                'Raster_Bildnummer': raster_bildnummer,
                'Database_Bildnummer': db_bildnummer,
                'Overlap_%': round(overlap_pct, 2)
            })
        
        # Create results dataframe
        self.results_df = pd.DataFrame(results)
        
        # Sort results
        self._sort_results()
        
        return self.results_df
    
    def _sort_results(self):
        """Sort results by SORTIE, then BestellID, then Raster_Bildnummer"""
        sort_columns = []
        if 'SORTIE' in self.results_df.columns:
            sort_columns.append('SORTIE')
        if 'BestellID' in self.results_df.columns:
            sort_columns.append('BestellID')
        if 'Raster_Bildnummer' in self.results_df.columns:
            sort_columns.append('Raster_Bildnummer')
        
        if sort_columns:
            self.results_df = self.results_df.sort_values(sort_columns, na_position='last').reset_index(drop=True)
    
    def display_results(self):
        """Display analysis results"""
        if self.results_df is None:
            print("No results to display. Please run analyze_overlap() first.")
            return
        
        print("Image Overlap Analysis Results (SORTIE-based matching)")
        print("=" * 80)
        print(f"\nTotal Raster Images: {len(self.results_df)}")
        matched_count = len(self.results_df[self.results_df['Database_Bildnummer'].notna()])
        print(f"Matched with Database: {matched_count} ({matched_count/len(self.results_df)*100:.1f}%)")
        
        print("\nResults Table:")
        print(self.results_df.to_string(index=False))
        print("\n" + "=" * 80)
    
    def display_statistics(self):
        """Display detailed statistics"""
        if self.results_df is None or self.results_df.empty:
            print("No results to analyze.")
            return
        
        matched_count = len(self.results_df[self.results_df['Database_Bildnummer'].notna()])
        
        print("\nMatching Statistics:")
        print(f"  Total images processed: {len(self.results_df)}")
        print(f"  Successfully matched: {matched_count}")
        print(f"  Match rate: {matched_count/len(self.results_df)*100:.1f}%")
        print(f"  Unmatched images: {len(self.results_df) - matched_count}")
        
        # Only show overlap statistics for matched images
        matched_df = self.results_df[self.results_df['Database_Bildnummer'].notna()]
        if not matched_df.empty:
            print("\nOverlap Statistics (matched images only):")
            print(f"  Average Overlap: {matched_df['Overlap_%'].mean():.2f}%")
            print(f"  Maximum Overlap: {matched_df['Overlap_%'].max():.2f}%")
            print(f"  Minimum Overlap: {matched_df['Overlap_%'].min():.2f}%")
            print(f"  Images with 0% overlap: {(matched_df['Overlap_%'] == 0).sum()}")
            print(f"  Images with >90% overlap: {(matched_df['Overlap_%'] > 90).sum()}")
    
    def save_results(self, output_path=None):
        """Save results to CSV file"""
        if self.results_df is None:
            print("No results to save. Please run analyze_overlap() first.")
            return
        
        if output_path is None:
            output_path = os.path.join(os.path.dirname(self.db_path), "overlap_analysis_results.csv")
        
        self.results_df.to_csv(output_path, index=False)
        print(f"Results saved to: {output_path}")
        return output_path
    
    def get_column_summary(self):
        """Display summary of identified columns"""
        print("Column Identification Summary:")
        print("-" * 40)
        print("Database Columns:")
        print(f"  BestellID: {self.bestellid_col_db}")
        print(f"  SORTIE: {self.sortie_col_db}")
        print(f"  Bildnummer: {self.bildnummer_col_db}")
        print(f"  Extent columns: {self.db_extent_cols}")
        
        print("\nRaster Columns:")
        print(f"  BestellID: {self.bestellid_col_raster}")
        print(f"  SORTIE: {self.sortie_col_raster}")
        print(f"  Bildnummer: {self.bildnummer_col_raster}")
        print(f"  Extent columns: {self.raster_extent_cols}")


# Main execution
if __name__ == "__main__":
    # Define paths
    db_path = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\DB_imageExtent.csv"
    raster_path = r"G:\0 BIPLOV\Scripts\JupyterNotebook_Scripts\REDIETBIPLOV - AutoGeorefImageCheck\Raster_ImageExtent.csv"
    
    # Create analyzer instance
    analyzer = ImageOverlapAnalyzer(db_path, raster_path)
    
    # Load data
    analyzer.load_data()
    
    # Display column summary
    analyzer.get_column_summary()
    
    # Run analysis
    results = analyzer.analyze_overlap()
    
    # Display results
    analyzer.display_results()
    
    # Display statistics
    analyzer.display_statistics()
    
    # Save results
    analyzer.save_results()

Column Identification Summary:
----------------------------------------
Database Columns:
  BestellID: bestellid
  SORTIE: sortie
  Bildnummer: bildnummer
  Extent columns: {'xmin': 'xmin', 'ymin': 'ymin', 'xmax': 'xmax', 'ymax': 'ymax'}

Raster Columns:
  BestellID: bestellid
  SORTIE: sortie
  Bildnummer: bildnummer
  Extent columns: {'xmin': 'xmin', 'ymin': 'ymin', 'xmax': 'xmax', 'ymax': 'ymax'}
Image Overlap Analysis Results (SORTIE-based matching)

Total Raster Images: 28
Matched with Database: 28 (100.0%)

Results Table:
 BestellID             SORTIE  Raster_Bildnummer  Database_Bildnummer  Overlap_%
    1742.0          106G_1837               3079               3079.0      93.00
    1742.0          106G_1837               3080               3080.0      92.18
    1739.0          106G_2338               3044               3044.0      92.50
    1737.0             2_3674               3242               3242.0      92.71
    1737.0             2_3674               3243             

## Georeferenced Image cross-check with Basemap

#### SAM (not good result)

##### Installing segment-anything model

In [ ]:
import sys
import subprocess

# Install SAM directly from GitHub
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/facebookresearch/segment-anything.git'])

print("SAM installed successfully from GitHub!")

##### Check for installation status

In [ ]:
import sys
import site

# Show where pip installs packages
print("Python executable:", sys.executable)
print("\nSite packages locations:")
for path in site.getsitepackages():
    print(f"  - {path}")

# Check if segment_anything is installed
try:
    import segment_anything
    print(f"\n✓ segment_anything is installed at: {segment_anything.__file__}")
except ImportError:
    print("\n✗ segment_anything is not installed")

##### Downloading SAM Checkpoint

In [ ]:
# Download SAM Checkpoint - CPU Optimized Version

import os
import requests
from pathlib import Path
import time

# Define paths
CHECKPOINT_DIR = Path(r"D:\FBM try\SAM Checkpoint")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# For CPU usage, use the smallest model (vit_b) to save memory and speed up processing
# Model options:
# - vit_b: ~375MB, fastest, lowest memory (RECOMMENDED FOR CPU)
# - vit_l: ~1.2GB, medium speed, medium memory
# - vit_h: ~2.4GB, slowest, highest memory (not recommended for CPU)

MODEL_CHOICES = {
    'vit_b': {
        'url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
        'filename': 'sam_vit_b_01ec64.pth',
        'size_mb': 375,
        'description': 'Fastest, lowest memory (Recommended for CPU)'
    },
    'vit_l': {
        'url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
        'filename': 'sam_vit_l_0b3195.pth',
        'size_mb': 1200,
        'description': 'Balanced speed and accuracy'
    },
    'vit_h': {
        'url': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        'filename': 'sam_vit_h_4b8939.pth',
        'size_mb': 2400,
        'description': 'Best accuracy, slowest (Not recommended for CPU)'
    }
}

# Select model - vit_b is recommended for CPU
SELECTED_MODEL = 'vit_h'  # Change to 'vit_l' or 'vit_h' if you have more RAM

model_info = MODEL_CHOICES[SELECTED_MODEL]
checkpoint_path = CHECKPOINT_DIR / model_info['filename']

print("=" * 70)
print("SAM CHECKPOINT DOWNLOADER")
print("=" * 70)
print(f"Selected model: {SELECTED_MODEL}")
print(f"Description: {model_info['description']}")
print(f"File size: ~{model_info['size_mb']} MB")
print(f"Save location: {checkpoint_path}")
print("=" * 70)

# Check if checkpoint already exists
if checkpoint_path.exists():
    file_size = checkpoint_path.stat().st_size / (1024 * 1024)
    print(f"\n✓ Checkpoint already exists at: {checkpoint_path}")
    print(f"  File size: {file_size:.1f} MB")
    
    # Verify file size is reasonable
    expected_size_mb = model_info['size_mb']
    if file_size < expected_size_mb * 0.8:  # If less than 80% of expected size
        print(f"  ⚠ Warning: File seems smaller than expected ({expected_size_mb} MB)")
        print("  Consider re-downloading.")
    else:
        print("  ✓ File size looks correct.")
else:
    print(f"\nDownloading {SELECTED_MODEL} checkpoint...")
    print(f"URL: {model_info['url']}")
    print("This may take several minutes depending on your internet speed...")
    
    try:
        # Download with progress tracking
        response = requests.get(model_info['url'], stream=True)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        
        with open(checkpoint_path, 'wb') as f:
            downloaded = 0
            start_time = time.time()
            
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    
                    # Print progress every 10%
                    if total_size > 0:
                        progress = (downloaded / total_size) * 100
                        if progress % 10 < 1:  # Print at 10% increments
                            elapsed = time.time() - start_time
                            speed = downloaded / (1024 * 1024 * elapsed) if elapsed > 0 else 0
                            print(f"  Progress: {progress:.1f}% ({downloaded/(1024*1024):.1f} MB) - Speed: {speed:.1f} MB/s", end='\r')
        
        print(f"\n\n✓ Download complete!")
        file_size = checkpoint_path.stat().st_size / (1024 * 1024)
        print(f"  File saved to: {checkpoint_path}")
        print(f"  File size: {file_size:.1f} MB")
        
    except requests.exceptions.RequestException as e:
        print(f"\n✗ Download failed: {e}")
        print("\nAlternative download methods:")
        print("  1. Download manually from:")
        print(f"     {model_info['url']}")
        print(f"  2. Save to: {checkpoint_path}")
        print("  3. Then run the segmentation pipeline.")

# Verify the checkpoint file
print("\n" + "=" * 70)
print("VERIFYING CHECKPOINT")
print("=" * 70)

if checkpoint_path.exists():
    import torch
    
    try:
        # Try to load the checkpoint to verify it's valid
        print("Loading checkpoint to verify integrity...")
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        print("✓ Checkpoint loaded successfully!")
        print(f"  Keys in checkpoint: {list(checkpoint.keys())}")
        
        # Check if it's a valid SAM checkpoint
        if 'model' in checkpoint or 'state_dict' in checkpoint:
            print("  ✓ Valid SAM checkpoint format detected")
        else:
            print("  ⚠ Unknown checkpoint format - may not be a SAM checkpoint")
            
    except Exception as e:
        print(f"✗ Error loading checkpoint: {e}")
        print("  The file may be corrupted. Try re-downloading.")
else:
    print("✗ Checkpoint file not found")

print("\n" + "=" * 70)
print("CHECKPOINT INFORMATION FOR PIPELINE")
print("=" * 70)
print(f"CHECKPOINT_PATH = r'{checkpoint_path}'")
print(f"MODEL_TYPE = '{SELECTED_MODEL}'  # Must match the downloaded model")
print(f"DEVICE = 'cpu'  # Using CPU for inference")
print("=" * 70)

##### Main Code

In [ ]:
# filename integrated_segmentation_pipeline.py

import os
import sys
import re
import warnings
import csv
import json
import gc
import torch
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features
from rasterio.mask import mask
from pathlib import Path
from collections import defaultdict
from shapely.geometry import Point, Polygon, MultiPolygon, shape
from shapely.ops import unary_union
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
from matplotlib import patches
import win32com.client
import pythoncom

# Add user site-packages to path if not already there
user_site_packages = Path(os.environ.get('APPDATA', '')) / 'Python' / 'Python312' / 'site-packages'
if user_site_packages.exists() and str(user_site_packages) not in sys.path:
    sys.path.insert(0, str(user_site_packages))
    print(f"Added user site-packages to path: {user_site_packages}")

# Segmentation libraries
try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor
    HAS_SAM = True
    print(f"SAM imported successfully")
except ImportError as e:
    HAS_SAM = False
    print(f"Segment-Anything not installed. Error: {e}")
    print("Install with: pip install --user git+https://github.com/facebookresearch/segment-anything.git")

warnings.filterwarnings('ignore')


class IntegratedGeorefSegmentationProcessor:
    """
    Integrated processor that finds georeferenced images from shortcuts
    and performs semantic segmentation using SAM.
    """
    
    def __init__(self, source_path, output_dir, checkpoint_path=None, 
                 model_type='vit_h', device='cpu', sam_kwargs=None):
        """
        Initialize the integrated processor.
        
        Args:
            source_path: Path to scan for shortcut files
            output_dir: Directory to save all outputs
            checkpoint_path: Path to SAM checkpoint file
            model_type: SAM model type ('vit_h', 'vit_l', 'vit_b')
            device: 'cuda' or 'cpu'
            sam_kwargs: Additional SAM arguments
        """
        # Initialize paths
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.checkpoint_path = checkpoint_path
        self.model_type = model_type
        self.device = device if torch.cuda.is_available() else 'cpu'
        
        # Create output subdirectories
        self.csv_dir = self.output_dir / 'csv_outputs'
        self.segmentation_dir = self.output_dir / 'segmentation_results'
        self.vector_dir = self.output_dir / 'vector_outputs'
        self.visualization_dir = self.output_dir / 'visualizations'
        self.metadata_dir = self.output_dir / 'metadata'
        
        for dir_path in [self.csv_dir, self.segmentation_dir, self.vector_dir,
                        self.visualization_dir, self.metadata_dir]:
            dir_path.mkdir(parents=True, exist_ok=True)
        
        # SAM configuration - optimized for memory and WWII aerial imagery
        self.sam_kwargs = sam_kwargs or {
            'points_per_side': 16,  # Reduced from 32 to save memory
            'pred_iou_thresh': 0.86,
            'stability_score_thresh': 0.92,
            'crop_n_layers': 0,  # Disabled cropping to save memory
            'crop_n_points_downscale_factor': 2,
            'min_mask_region_area': 100,
            'output_mode': 'binary_mask'
        }
        
        # Image downsampling parameters for memory management
        self.max_image_size = 1024  # Maximum dimension for SAM input
        self.tile_size = 512  # Size for tiling large images
        self.overlap = 64  # Overlap between tiles
        
        # Initialize SAM model
        self.sam = None
        self.mask_generator = None
        self.predictor = None
        
        # Check if SAM is available and checkpoint exists
        if HAS_SAM and checkpoint_path:
            checkpoint_path = Path(checkpoint_path)
            if checkpoint_path.exists():
                self._initialize_sam()
            else:
                print(f"Warning: Checkpoint file not found at {checkpoint_path}")
                print("Please download from: https://dl.fbaipublicfiles.com/segment_anything/")
                print(f"For model type '{model_type}', download the corresponding checkpoint.")
        else:
            if not HAS_SAM:
                print("Warning: SAM library not available")
            if not checkpoint_path:
                print("Warning: No checkpoint path provided")
        
        # Data storage - same as original GeorefImageProcessor
        self.shortcuts = []
        self.shortcuts_with_targets = []
        self.raster_data = []
        self.all_images = []
        self.image_data_df = None
        self.georeferenced_image_paths = []
        
        print(f"\n{'='*60}")
        print(f"INITIALIZED PROCESSOR")
        print(f"{'='*60}")
        print(f"Source path: {self.source_path}")
        print(f"Output directory: {self.output_dir}")
        print(f"Device: {self.device}")
        print(f"SAM available: {HAS_SAM and self.sam is not None}")
        print(f"Checkpoint: {self.checkpoint_path}")
        print(f"Max image size: {self.max_image_size}px")
        print(f"{'='*60}\n")
    
    def _initialize_sam(self):
        """Initialize the SAM model."""
        try:
            print(f"Loading SAM model: {self.model_type}")
            print(f"Checkpoint: {self.checkpoint_path}")
            
            # Load the model
            self.sam = sam_model_registry[self.model_type](checkpoint=self.checkpoint_path)
            self.sam.to(device=self.device)
            self.sam.eval()
            
            # Create mask generator with optimized settings
            self.mask_generator = SamAutomaticMaskGenerator(
                model=self.sam,
                **self.sam_kwargs
            )
            
            # Create predictor for potential interactive use
            self.predictor = SamPredictor(self.sam)
            
            print(f"SAM model initialized successfully on {self.device}")
            
        except Exception as e:
            print(f"Error initializing SAM: {e}")
            print("Possible issues:")
            print("  1. Checkpoint file may be corrupted or incompatible")
            print("  2. Model type may not match checkpoint")
            print("  3. Insufficient memory for the model")
            self.sam = None
            self.mask_generator = None
            self.predictor = None
    
    # ==================== GEOREFERENCING METHODS ====================
    
    def parse_image_info(self, filename):
        """Parse filename pattern and extract components."""
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').replace('_Finalized', '')
        parts = [p.strip() for p in filename.split('#') if p.strip()]
        
        image_num = flight_num = sortie = bestell_id = date = None
        
        if parts:
            image_num = parts[0]
        
        for part in parts:
            if part.startswith('ID') and any(c.isdigit() for c in part[2:]):
                numbers = re.findall(r'\d+', part)
                bestell_id = numbers[0] if numbers else None
                break
        
        for part in parts:
            if re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                date = part
                break
        
        if len(parts) >= 3:
            for part in parts[1:-1]:
                if part != bestell_id and not re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                    if not flight_num:
                        flight_num = part
                    elif not sortie:
                        sortie = part
        
        return {
            'image_number': image_num,
            'flight_number': flight_num,
            'sortie': sortie,
            'bestell_id': bestell_id,
            'date': date,
            'original_name': filename
        }
    
    def find_and_process_shortcuts(self):
        """Find all .lnk files and parse their information."""
        print(f"\nScanning for shortcut files in {self.source_path}...")
        
        for root, dirs, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    file_without_ext = file[:-4]
                    parsed_info = self.parse_image_info(file_without_ext)
                    parsed_info['full_path'] = Path(root) / file
                    parsed_info['folder'] = root
                    self.shortcuts.append(parsed_info)
        
        print(f"Found {len(self.shortcuts)} shortcut files")
        return self.shortcuts
    
    def get_shortcut_target(self, lnk_path):
        """Extract target path from Windows shortcut."""
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target_path = shortcut.TargetPath
            pythoncom.CoUninitialize()
            return target_path if target_path else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path}: {e}")
            return None
    
    def find_modified_versions(self, original_path):
        """Look for modified versions of an original image."""
        if not original_path or not original_path.exists():
            return []
        
        folder = original_path.parent
        stem = original_path.stem
        modified_patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        modified_files = []
        
        for pattern in modified_patterns:
            potential_file = folder / f"{stem}{pattern}{original_path.suffix}"
            if potential_file.exists():
                modified_files.append(potential_file)
        
        for file in folder.iterdir():
            if file.is_file() and original_path.suffix == file.suffix:
                file_stem = file.stem
                if (file_stem.startswith(stem) and 
                    any(pattern in file_stem for pattern in modified_patterns)):
                    modified_files.append(file)
        
        return list(set(modified_files))
    
    def extract_shortcut_targets(self):
        """Extract target paths from all shortcuts and find modified versions."""
        print("\nExtracting shortcut targets and finding modified versions...")
        
        for shortcut_data in self.shortcuts:
            lnk_path = shortcut_data['full_path']
            target_path = self.get_shortcut_target(lnk_path)
            
            if target_path:
                target_path = Path(target_path)
                shortcut_data.update({
                    'target_path': target_path,
                    'target_exists': target_path.exists(),
                    'target_folder': target_path.parent,
                    'target_filename': target_path.name
                })
                
                target_info = self.parse_image_info(target_path.stem)
                shortcut_data.update({
                    'target_image_number': target_info['image_number'],
                    'target_bestell_id': target_info['bestell_id'],
                    'target_sortie': target_info['sortie'],
                    'target_date': target_info['date']
                })
                
                if shortcut_data['target_exists']:
                    modified_files = self.find_modified_versions(target_path)
                    if modified_files:
                        shortcut_data['modified_files'] = modified_files
                
                self.shortcuts_with_targets.append(shortcut_data)
        
        print(f"Successfully processed {len(self.shortcuts_with_targets)} shortcuts")
        return self.shortcuts_with_targets
    
    def calculate_tiff_area_extent(self, tiff_path):
        """Calculate area and geographic extent of TIFF file."""
        try:
            with rasterio.open(tiff_path) as src:
                rows, cols = src.height, src.width
                transform = src.transform
                pixel_res_x, pixel_res_y = transform.a, -transform.e
                bounds = src.bounds
                
                pixel_area = abs(pixel_res_x * pixel_res_y)
                total_area = pixel_area * rows * cols
                
                return {
                    'area_m2': float(total_area),
                    'rows': int(rows),
                    'cols': int(cols),
                    'pixel_res_x_m': float(pixel_res_x),
                    'pixel_res_y_m': float(pixel_res_y),
                    'bounds': bounds,
                    'is_georeferenced': src.crs is not None,
                    'crs': src.crs,
                    'transform': transform,
                    'image_data': src.read(1) if src.count >= 1 else None
                }
        except Exception as e:
            print(f"Error processing TIFF {tiff_path.name}: {e}")
            return None
    
    def process_georeferenced_images(self):
        """Process all georeferenced TIFF files and collect data."""
        print("\nProcessing georeferenced TIFF files...")
        self.georeferenced_image_paths = []
        
        for shortcut in self.shortcuts_with_targets:
            bestell_id = shortcut.get('bestell_id') or shortcut.get('target_bestell_id')
            bestell_id_float = float(bestell_id) if bestell_id and bestell_id.isdigit() else 0.0
            
            base_record = {
                'BestellID': bestell_id_float,
                'SORTIE': shortcut.get('sortie') or shortcut.get('target_sortie') or '',
                'Bildnummer': shortcut.get('image_number') or shortcut.get('target_image_number') or '',
                'Rows': 0.0,
                'Cols': 0.0,
                'Res_m': 0.0,
                'Area_sqm': 0.0,
                'X_min': 0.0,
                'Y_min': 0.0,
                'X_max': 0.0,
                'Y_max': 0.0,
                'Has_Georef': False,
                'Image_Name': shortcut.get('target_filename', ''),
                'Image_Path': str(shortcut.get('target_path', ''))
            }
            
            has_georef_version = False
            
            if 'modified_files' in shortcut:
                for modified_file in shortcut['modified_files']:
                    if modified_file.suffix.lower() in ['.tif', '.tiff']:
                        area_info = self.calculate_tiff_area_extent(modified_file)
                        
                        if area_info and area_info['is_georeferenced']:
                            has_georef_version = True
                            record = base_record.copy()
                            record.update({
                                'Rows': float(area_info['rows']),
                                'Cols': float(area_info['cols']),
                                'Res_m': float((area_info['pixel_res_x_m'] + area_info['pixel_res_y_m']) / 2),
                                'Area_sqm': float(area_info['area_m2']),
                                'X_min': float(area_info['bounds'].left),
                                'Y_min': float(area_info['bounds'].bottom),
                                'X_max': float(area_info['bounds'].right),
                                'Y_max': float(area_info['bounds'].top),
                                'Has_Georef': True,
                                'Image_Name': modified_file.name,
                                'Image_Path': str(modified_file),
                                'crs': area_info['crs'],
                                'transform': area_info['transform']
                            })
                            self.raster_data.append(record)
                            self.georeferenced_image_paths.append(modified_file)
            
            if has_georef_version:
                georef_records = [r for r in self.raster_data if r['Image_Name'] in 
                                 [f.name for f in shortcut.get('modified_files', [])]]
                self.all_images.extend(georef_records)
            else:
                base_record['Has_Georef'] = False
                self.all_images.append(base_record)
        
        # Create DataFrame from georeferenced images
        if self.raster_data:
            self.image_data_df = pd.DataFrame(self.raster_data)
            print(f"Processed {len(self.raster_data)} georeferenced images")
            print(f"Found {len(self.georeferenced_image_paths)} unique georeferenced TIFF files")
        else:
            print("No georeferenced images found!")
            self.image_data_df = pd.DataFrame()
        
        return self.raster_data, self.all_images
    
    def save_georef_csv(self):
        """Save the georeferencing results to CSV."""
        if not self.all_images:
            return None
        
        output_path = self.csv_dir / "Raster_ImageExtent.csv"
        
        # Remove any extra columns added for internal use
        data_to_save = []
        for record in self.all_images:
            clean_record = {k: v for k, v in record.items() 
                          if k not in ['Image_Path', 'crs', 'transform']}
            data_to_save.append(clean_record)
        
        with open(output_path, 'w', newline='', encoding='utf-8') as csvfile:
            fieldnames = ['BestellID', 'SORTIE', 'Bildnummer', 'Rows', 'Cols', 'Res_m', 
                         'Area_sqm', 'X_min', 'Y_min', 'X_max', 'Y_max', 'Has_Georef', 'Image_Name']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            for record in data_to_save:
                writer.writerow(record)
        
        print(f"Georeferencing CSV saved to: {output_path}")
        return output_path
    
    # ==================== SEGMENTATION METHODS ====================
    
    def read_and_downsample_geotiff(self, image_path):
        """Read a georeferenced TIFF and downsample if too large."""
        try:
            with rasterio.open(image_path) as src:
                # Get original dimensions
                height, width = src.height, src.width
                
                # Calculate downsampling factor if needed
                scale_factor = 1.0
                if max(height, width) > self.max_image_size:
                    scale_factor = self.max_image_size / max(height, width)
                
                # Read the first band
                if src.count >= 1:
                    if scale_factor < 1.0:
                        # Read at reduced resolution
                        new_height = int(height * scale_factor)
                        new_width = int(width * scale_factor)
                        out_shape = (new_height, new_width)
                        image = src.read(1, out_shape=out_shape)
                        # Update transform for downsampled image
                        transform = src.transform * src.transform.scale(
                            (width) / new_width,
                            (height) / new_height
                        )
                    else:
                        image = src.read(1)
                        transform = src.transform
                else:
                    return None
                
                # Normalize to 8-bit
                if image.dtype != np.uint8:
                    image = np.nan_to_num(image, nan=0, posinf=255, neginf=0)
                    if image.max() > image.min():
                        image_norm = (image - image.min()) / (image.max() - image.min()) * 255
                        image_8bit = image_norm.astype(np.uint8)
                    else:
                        image_8bit = np.zeros_like(image, dtype=np.uint8)
                else:
                    image_8bit = image
                
                # Convert to RGB for SAM (3 channels)
                image_rgb = np.stack([image_8bit] * 3, axis=2)
                
                return {
                    'image': image_rgb,
                    'image_original': image_8bit,
                    'profile': src.profile.copy(),
                    'bounds': src.bounds,
                    'transform': transform,
                    'crs': src.crs,
                    'height': image_rgb.shape[0],
                    'width': image_rgb.shape[1],
                    'original_height': height,
                    'original_width': width,
                    'scale_factor': scale_factor,
                    'path': image_path
                }
        except Exception as e:
            print(f"Error reading {image_path}: {e}")
            return None
    
    def apply_clahe_enhancement(self, image, clip_limit=2.0, tile_grid_size=(8, 8)):
        """Apply CLAHE contrast enhancement for WWII imagery."""
        try:
            if len(image.shape) == 3:
                lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
                l, a, b = cv2.split(lab)
                clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
                l_enhanced = clahe.apply(l)
                lab_enhanced = cv2.merge([l_enhanced, a, b])
                enhanced = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB)
            else:
                clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
                enhanced = clahe.apply(image)
            return enhanced
        except Exception as e:
            print(f"Error applying CLAHE enhancement: {e}")
            return image
    
    def generate_masks_sam(self, image_rgb, image_info):
        """Generate masks using SAM automatic mask generation with memory optimization."""
        if self.mask_generator is None:
            print("SAM mask generator not available")
            return []
        
        try:
            # Apply enhancement for better feature extraction
            enhanced_image = self.apply_clahe_enhancement(image_rgb)
            
            # Ensure image is not too large for SAM
            h, w = enhanced_image.shape[:2]
            if h > self.max_image_size or w > self.max_image_size:
                # Downsample further if needed
                scale = min(self.max_image_size / h, self.max_image_size / w)
                new_h, new_w = int(h * scale), int(w * scale)
                enhanced_image = cv2.resize(enhanced_image, (new_w, new_h), 
                                          interpolation=cv2.INTER_AREA)
                print(f"  Downsampled image from {h}x{w} to {new_h}x{new_w} for SAM")
            
            # Generate masks
            masks = self.mask_generator.generate(enhanced_image)
            
            # Add image metadata to each mask
            for mask in masks:
                mask['image_path'] = str(image_info['path'])
                mask['image_bounds'] = image_info['bounds']
                mask['image_transform'] = image_info['transform']
                mask['image_crs'] = image_info['crs']
                mask['image_width'] = image_info['original_width']
                mask['image_height'] = image_info['original_height']
                mask['scale_factor'] = image_info['scale_factor']
            
            return masks
        except Exception as e:
            print(f"Error generating masks: {e}")
            return []
    
    def mask_to_polygon(self, mask, transform, crs):
        """Convert binary mask to Shapely polygon with georeferencing."""
        try:
            shapes = features.shapes(
                mask.astype(np.uint8),
                mask=mask,
                transform=transform
            )
            
            polygons = []
            for geom, value in shapes:
                if value == 1:
                    polygon = shape(geom)
                    if polygon.is_valid and not polygon.is_empty:
                        # Simplify to reduce complexity
                        polygon = polygon.simplify(1.0)
                        polygons.append(polygon)
            
            if polygons:
                combined = unary_union(polygons)
                return combined
            return None
        except Exception as e:
            print(f"Error converting mask to polygon: {e}")
            return None
    
    def calculate_mask_properties(self, mask, transform, crs, scale_factor=1.0):
        """Calculate properties for a segmented mask."""
        try:
            # Scale factor to convert mask pixels back to original resolution if needed
            if scale_factor < 1.0:
                # Resize mask back to original dimensions for accurate area
                mask = cv2.resize(mask.astype(np.uint8), 
                                (0, 0), 
                                fx=1/scale_factor, 
                                fy=1/scale_factor,
                                interpolation=cv2.INTER_NEAREST)
                mask = mask > 0
            
            pixel_area = np.sum(mask)
            polygon = self.mask_to_polygon(mask, transform, crs)
            
            if polygon:
                geographic_area = polygon.area
                bounds = polygon.bounds
                centroid = polygon.centroid
            else:
                geographic_area = 0
                bounds = (0, 0, 0, 0)
                centroid = Point(0, 0)
            
            rows = np.any(mask, axis=1)
            cols = np.any(mask, axis=0)
            if np.any(rows) and np.any(cols):
                rmin, rmax = np.where(rows)[0][[0, -1]]
                cmin, cmax = np.where(cols)[0][[0, -1]]
                bbox_pixels = (cmin, rmin, cmax, rmax)
                bbox_width_pixels = cmax - cmin
                bbox_height_pixels = rmax - rmin
            else:
                bbox_pixels = (0, 0, 0, 0)
                bbox_width_pixels = 0
                bbox_height_pixels = 0
            
            return {
                'pixel_area': int(pixel_area),
                'geographic_area': float(geographic_area),
                'geographic_area_ha': float(geographic_area / 10000) if geographic_area > 0 else 0,
                'bounds': bounds,
                'centroid': centroid,
                'bbox_pixels': bbox_pixels,
                'bbox_width_pixels': int(bbox_width_pixels),
                'bbox_height_pixels': int(bbox_height_pixels),
                'aspect_ratio': (bbox_width_pixels / bbox_height_pixels) if bbox_height_pixels > 0 else 0,
                'has_polygon': polygon is not None,
                'polygon': polygon if polygon else None
            }
        except Exception as e:
            print(f"Error calculating mask properties: {e}")
            return None
    
    def classify_features(self, mask, image_rgb, properties):
        """Classify segmented features based on shape, size, and intensity."""
        try:
            # Extract intensity from masked region
            if len(image_rgb.shape) == 3:
                gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
                # Ensure mask is same size as image
                if mask.shape != gray.shape:
                    mask = cv2.resize(mask.astype(np.uint8), 
                                    (gray.shape[1], gray.shape[0]),
                                    interpolation=cv2.INTER_NEAREST)
                    mask = mask > 0
                masked_pixels = gray[mask]
                mean_intensity = np.mean(masked_pixels) if len(masked_pixels) > 0 else 0
            else:
                masked_pixels = image_rgb[mask]
                mean_intensity = np.mean(masked_pixels) if len(masked_pixels) > 0 else 0
            
            area_ha = properties['geographic_area_ha']
            aspect_ratio = properties['aspect_ratio']
            
            # Enhanced classification for WWII aerial imagery
            if area_ha > 10 and aspect_ratio > 3:
                return 'linear_feature_large'
            elif area_ha > 5 and aspect_ratio > 2:
                return 'linear_feature_medium'
            elif area_ha > 0.5 and aspect_ratio < 1.5:
                if mean_intensity < 80:
                    return 'building_dark'
                else:
                    return 'building_light'
            elif area_ha > 0.1 and aspect_ratio < 1.5:
                return 'small_structure'
            elif area_ha > 5:
                if mean_intensity < 100:
                    return 'water_body'
                else:
                    return 'open_area'
            elif area_ha > 0.05:
                return 'minor_feature'
            else:
                return 'unknown_feature'
        except Exception as e:
            print(f"Error classifying feature: {e}")
            return 'unknown_feature'
    
    def process_single_image(self, image_row):
        """Process a single image for segmentation."""
        try:
            image_path = Path(image_row['Image_Path'])
            
            if not image_path.exists():
                print(f"Image not found: {image_path}")
                return None
            
            print(f"\nProcessing: {image_path.name}")
            
            # Read and downsample the image if needed
            image_info = self.read_and_downsample_geotiff(image_path)
            if image_info is None:
                return None
            
            print(f"  Image size: {image_info['height']}x{image_info['width']}")
            if image_info['scale_factor'] < 1.0:
                print(f"  Downsampled by factor: {image_info['scale_factor']:.2f}")
            
            # Generate masks
            masks = self.generate_masks_sam(image_info['image'], image_info)
            
            if not masks:
                print(f"  No masks generated for {image_path.name}")
                return []
            
            # Process each mask
            results = []
            for idx, mask_data in enumerate(masks):
                mask = mask_data['segmentation']
                
                properties = self.calculate_mask_properties(
                    mask, 
                    image_info['transform'], 
                    image_info['crs'],
                    image_info['scale_factor']
                )
                
                if properties is None or not properties['has_polygon']:
                    continue
                
                # Skip very small features (noise)
                if properties['pixel_area'] < 50:
                    continue
                
                classification = self.classify_features(mask, image_info['image'], properties)
                
                result = {
                    'image_name': image_path.name,
                    'image_path': str(image_path),
                    'mask_id': idx,
                    'bestell_id': image_row.get('BestellID', 0),
                    'sortie': image_row.get('SORTIE', ''),
                    'bildnummer': image_row.get('Bildnummer', ''),
                    'classification': classification,
                    'pixel_area': properties['pixel_area'],
                    'area_sqm': properties['geographic_area'],
                    'area_ha': properties['geographic_area_ha'],
                    'centroid_x': properties['centroid'].x,
                    'centroid_y': properties['centroid'].y,
                    'bbox_xmin': properties['bbox_pixels'][0],
                    'bbox_ymin': properties['bbox_pixels'][1],
                    'bbox_xmax': properties['bbox_pixels'][2],
                    'bbox_ymax': properties['bbox_pixels'][3],
                    'aspect_ratio': properties['aspect_ratio'],
                    'bounds_minx': properties['bounds'][0],
                    'bounds_miny': properties['bounds'][1],
                    'bounds_maxx': properties['bounds'][2],
                    'bounds_maxy': properties['bounds'][3],
                    'crs': str(image_info['crs']) if image_info['crs'] else None,
                    'geometry': properties['polygon']
                }
                
                results.append(result)
            
            # Save segmentation results for this image
            if results:
                self.save_segmentation_results(image_path.name, results)
                print(f"  ✓ Found {len(results)} segments")
            else:
                print(f"  ○ No valid segments")
            
            return results
            
        except Exception as e:
            print(f"Error processing {image_row.get('Image_Name', 'unknown')}: {e}")
            import traceback
            traceback.print_exc()
            return None
    
    def save_segmentation_results(self, image_name, results):
        """Save segmentation results to files."""
        try:
            image_stem = Path(image_name).stem
            
            # Create GeoDataFrame
            gdf = self.results_to_geodataframe(results)
            if gdf is not None and len(gdf) > 0:
                # Save as GeoPackage
                gpkg_path = self.vector_dir / f"{image_stem}_segments.gpkg"
                gdf.to_file(gpkg_path, driver='GPKG')
                
                # Save as GeoJSON
                geojson_path = self.vector_dir / f"{image_stem}_segments.geojson"
                gdf.to_file(geojson_path, driver='GeoJSON')
            
            # Save metadata
            metadata_path = self.metadata_dir / f"{image_stem}_metadata.json"
            metadata = {
                'image_name': image_name,
                'number_of_segments': len(results),
                'segment_classes': list(set(r['classification'] for r in results)),
                'total_area_ha': sum(r['area_ha'] for r in results),
                'class_counts': {cls: sum(1 for r in results if r['classification'] == cls) 
                               for cls in set(r['classification'] for r in results)}
            }
            with open(metadata_path, 'w') as f:
                json.dump(metadata, f, indent=2, default=str)
                
        except Exception as e:
            print(f"Error saving results for {image_name}: {e}")
    
    def results_to_geodataframe(self, results):
        """Convert segmentation results to GeoDataFrame."""
        try:
            geometries = []
            for r in results:
                if 'geometry' in r and r['geometry'] is not None:
                    geometries.append(r['geometry'])
                else:
                    geometries.append(Point(r['centroid_x'], r['centroid_y']))
            
            if geometries:
                results_copy = []
                for r in results:
                    r_copy = {k: v for k, v in r.items() if k != 'geometry'}
                    results_copy.append(r_copy)
                
                gdf = gpd.GeoDataFrame(results_copy, geometry=geometries)
                if 'crs' in results[0] and results[0]['crs']:
                    try:
                        gdf.crs = results[0]['crs']
                    except:
                        pass
                return gdf
            
            return None
        except Exception as e:
            print(f"Error creating GeoDataFrame: {e}")
            return None
    
    def visualize_segmentation(self, image_path, results, output_path):
        """Create visualization of segmentation results."""
        try:
            image_info = self.read_and_downsample_geotiff(image_path)
            if image_info is None:
                return
            
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Original image
            axes[0].imshow(image_info['image'], cmap='gray')
            axes[0].set_title('Original Image')
            axes[0].axis('off')
            
            # Classified features
            axes[1].imshow(image_info['image'], cmap='gray')
            colors = {
                'building_dark': 'blue',
                'building_light': 'cyan',
                'linear_feature_large': 'green',
                'linear_feature_medium': 'lime',
                'water_body': 'deepskyblue',
                'open_area': 'yellow',
                'small_structure': 'orange',
                'minor_feature': 'purple',
                'unknown_feature': 'gray'
            }
            
            for r in results:
                bbox = [r['bbox_xmin'], r['bbox_ymin'], 
                       r['bbox_xmax'], r['bbox_ymax']]
                rect = patches.Rectangle(
                    (bbox[0], bbox[1]), 
                    bbox[2] - bbox[0], 
                    bbox[3] - bbox[1],
                    linewidth=1, 
                    edgecolor=colors.get(r['classification'], 'red'),
                    facecolor='none'
                )
                axes[1].add_patch(rect)
                axes[1].text(
                    bbox[0], bbox[1] - 5, 
                    r['classification'][:10],
                    fontsize=6,
                    color='white',
                    backgroundcolor='black'
                )
            
            axes[1].set_title('Classified Features')
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.savefig(output_path, dpi=150, bbox_inches='tight')
            plt.close()
            
        except Exception as e:
            print(f"Error creating visualization: {e}")
    
    def process_all_images_for_segmentation(self, max_images=None, save_visualizations=True):
        """Process all georeferenced images for segmentation."""
        if self.image_data_df is None or len(self.image_data_df) == 0:
            print("No georeferenced images to process")
            return {}
        
        if self.sam is None:
            print("SAM model not initialized. Cannot perform segmentation.")
            print("Please provide a valid checkpoint path.")
            return {}
        
        # Limit number of images
        images_to_process = self.image_data_df
        if max_images:
            images_to_process = images_to_process.head(max_images)
        
        print(f"\n{'='*60}")
        print(f"PROCESSING {len(images_to_process)} IMAGES FOR SEGMENTATION")
        print(f"{'='*60}")
        
        all_results = []
        processed_count = 0
        failed_count = 0
        
        for idx, row in tqdm(images_to_process.iterrows(), total=len(images_to_process), desc="Segmenting images"):
            results = self.process_single_image(row)
            
            if results:
                all_results.extend(results)
                processed_count += 1
                
                if save_visualizations:
                    image_name = row['Image_Name']
                    image_stem = Path(image_name).stem
                    vis_path = self.visualization_dir / f"{image_stem}_segmentation.png"
                    self.visualize_segmentation(
                        Path(row['Image_Path']),
                        results,
                        vis_path
                    )
            else:
                failed_count += 1
            
            # Clear GPU/CPU memory
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
        
        # Save aggregated results
        if all_results:
            self.save_aggregated_results(all_results)
        
        summary = {
            'total_images': len(images_to_process),
            'processed_successfully': processed_count,
            'failed': failed_count,
            'total_segments': len(all_results),
            'segment_classes': self.get_class_distribution(all_results),
            'total_area_ha': sum(r['area_ha'] for r in all_results) if all_results else 0
        }
        
        # Save summary
        summary_path = self.output_dir / 'segmentation_summary.json'
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        
        print("\n" + "=" * 60)
        print("SEGMENTATION PROCESSING COMPLETE")
        print("=" * 60)
        print(f"Images processed: {processed_count}/{len(images_to_process)}")
        print(f"Total segments found: {len(all_results)}")
        print(f"Total area segmented: {summary['total_area_ha']:.2f} ha")
        print(f"\nClass distribution:")
        for cls, count in sorted(summary['segment_classes'].items(), key=lambda x: x[1], reverse=True):
            print(f"  {cls}: {count} segments")
        
        return summary
    
    def get_class_distribution(self, results):
        """Calculate class distribution from results."""
        class_counts = {}
        for r in results:
            cls = r.get('classification', 'unknown')
            class_counts[cls] = class_counts.get(cls, 0) + 1
        return class_counts
    
    def save_aggregated_results(self, all_results):
        """Save all results to CSV and GeoPackage."""
        try:
            # Save CSV
            df = pd.DataFrame(all_results)
            df_csv = df.drop(columns=['geometry'], errors='ignore')
            csv_path = self.output_dir / 'all_segmentation_results.csv'
            df_csv.to_csv(csv_path, index=False)
            print(f"Aggregated results saved to: {csv_path}")
            
            # Save GeoPackage
            if 'geometry' in df.columns:
                gdf = gpd.GeoDataFrame(df, geometry='geometry')
                if 'crs' in df.columns and df['crs'].iloc[0]:
                    try:
                        gdf.crs = df['crs'].iloc[0]
                    except:
                        pass
                gpkg_path = self.output_dir / 'all_segmentation_results.gpkg'
                gdf.to_file(gpkg_path, driver='GPKG')
                print(f"Aggregated GeoPackage saved to: {gpkg_path}")
                
        except Exception as e:
            print(f"Error saving aggregated results: {e}")
    
    # ==================== MAIN RUN METHOD ====================
    
    def run(self, max_images_for_segmentation=None, skip_segmentation=False):
        """
        Run the complete integrated pipeline.
        
        Args:
            max_images_for_segmentation: Max images for segmentation (None for all)
            skip_segmentation: If True, only run georeferencing, skip segmentation
        
        Returns:
            Dictionary with pipeline results
        """
        print("\n" + "=" * 80)
        print("INTEGRATED GEOREFERENCE + SEGMENTATION PIPELINE")
        print("=" * 80)
        
        # Step 1: Find and process shortcuts (georeferencing)
        self.find_and_process_shortcuts()
        self.extract_shortcut_targets()
        self.process_georeferenced_images()
        
        # Step 2: Save georeferencing CSV
        csv_path = self.save_georef_csv()
        
        # Step 3: Display summary of georeferenced images
        print("\n" + "=" * 60)
        print("GEOREFERENCING SUMMARY")
        print("=" * 60)
        print(f"Total shortcuts found: {len(self.shortcuts)}")
        print(f"Shortcuts with valid targets: {len(self.shortcuts_with_targets)}")
        print(f"Georeferenced images found: {len(self.raster_data)}")
        
        # Step 4: Run segmentation if requested
        segmentation_results = {}
        if not skip_segmentation and HAS_SAM and self.sam is not None:
            segmentation_results = self.process_all_images_for_segmentation(
                max_images=max_images_for_segmentation
            )
        elif skip_segmentation:
            print("\nSegmentation skipped as requested.")
        else:
            print("\nSegmentation skipped - SAM not available.")
            if not HAS_SAM:
                print("  Reason: Segment-Anything library not installed")
            if self.sam is None:
                print("  Reason: SAM model not initialized (check checkpoint path)")
        
        # Final summary
        print("\n" + "=" * 80)
        print("PIPELINE COMPLETE")
        print("=" * 80)
        print(f"Output directory: {self.output_dir}")
        print(f"Georeferencing CSV: {csv_path}")
        if segmentation_results:
            print(f"Segmentation completed: {segmentation_results.get('processed_successfully', 0)} images")
            print(f"Total segments: {segmentation_results.get('total_segments', 0)}")
        
        return {
            'georeferencing': {
                'shortcuts_count': len(self.shortcuts),
                'georef_images_count': len(self.raster_data),
                'csv_path': str(csv_path) if csv_path else None
            },
            'segmentation': segmentation_results
        }


# ==================== MAIN EXECUTION ====================

if __name__ == "__main__":
    # Configuration
    SOURCE_PATH = r"D:\FBM try\shorcuts"
    OUTPUT_DIR = r"D:\FBM try\outputs"
    
    # SAM Configuration
    CHECKPOINT_PATH = r"D:\FBM try\SAM Checkpoint\sam_vit_b_01ec64.pth"
    MODEL_TYPE = 'vit_b'
    DEVICE = 'cpu'
    
    print(f"\nPython executable: {sys.executable}")
    print(f"SAM installed: {HAS_SAM}")
    
    # Create and run integrated processor
    processor = IntegratedGeorefSegmentationProcessor(
        source_path=SOURCE_PATH,
        output_dir=OUTPUT_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        model_type=MODEL_TYPE,
        device=DEVICE
    )
    
    # Run the complete pipeline
    results = processor.run(
        max_images_for_segmentation=2,
        skip_segmentation=False
    )
    
    print(f"\nPipeline completed successfully!")
    print(f"Results saved to: {OUTPUT_DIR}")

#### Frangi ridge filter, run for both bright and dark ridges (not good results)

In [ ]:
# pip install scikit-image

In [ ]:
# filename: georef_linear_feature_pipeline_optimized.py
#
# Optimized Pipeline for extracting ROAD / RIVER / RAILWAY candidates as polygons
# from georeferenced WWII aerial imagery, with performance enhancements.
#
# PERFORMANCE FIXES vs. the previous "optimized" version
# --------------------------------------------------------------------------
# 1) REAL tile parallelism. Previously, `_preprocess` (denoise + CLAHE) and
#    `_ridge_response` (double Frangi pass) — the two most expensive steps —
#    were executed serially in the main process BEFORE the tile was ever
#    handed to the worker pool. The worker only did a percentile threshold,
#    which is essentially free. So "parallel" mode did all the expensive
#    work single-threaded and then parallelized almost nothing.
#    This also meant `pool.submit(self._worker_process_tile, ...)` had to
#    pickle a bound method of `self`, and `self` holds a `threading.Lock`
#    (`self.progress_lock`) — locks are not picklable, so this would raise
#    at runtime on Windows (spawn-based multiprocessing) as soon as it ran.
#    Fixed by moving denoise + CLAHE + Frangi + threshold into a
#    module-level worker function that only receives plain, picklable data.
#
# 2) Tiles no longer get pickled/copied whole through the task queue.
#    The full-resolution band is put into a `multiprocessing.shared_memory`
#    block once per image; each worker slices its own tile directly out of
#    shared memory. With overlapping tiles this avoids repeatedly
#    serializing the same overlapping pixels through IPC.
#
# 3) The worker pool is created once and reused across all images instead
#    of being spun up and torn down per image (process spawn is expensive).
#
# 4) Fixed a crash bug: the previous code did
#       del band, ridge_mask, wide_water_mask
#       return {'ridge_mask': ridge_mask, 'wide_water_mask': wide_water_mask,
#               'band': band, ...}
#    i.e. it deleted the local names and then referenced them in the return
#    dict on the very next lines — this raises UnboundLocalError. Removed
#    the premature `del`; these arrays are still needed by the caller.
#
# 5) `_detect_wide_water` claimed to use `cv2.boxFilter` in a comment but
#    actually called `cv2.filter2D` with a manually built ones-kernel, and
#    computed `img_f ** 2` as a separate full-size array. Replaced with the
#    real `cv2.boxFilter` (integral-image based, O(1) per pixel regardless
#    of window size) and `cv2.sqrBoxFilter` for the sum-of-squares term.
#
# 6) Frangi now runs on float32 instead of float64 (half the memory
#    bandwidth, faster, negligible precision impact for this use case).
#
# All detection thresholds, filtering, classification, and OSM-refinement
# logic are unchanged from the previous version.

import os
import re
import json
import csv
import warnings
import gc
from pathlib import Path
from multiprocessing import shared_memory
from concurrent.futures import ProcessPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool
from threading import Lock

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio import features as rio_features
from rasterio import Affine
from shapely.geometry import shape
from shapely.ops import unary_union
from scipy import ndimage as ndi
from skimage.morphology import skeletonize, remove_small_objects, binary_closing, disk
from skimage.measure import label, regionprops
import cv2
from tqdm import tqdm
import win32com.client
import pythoncom

# The tile worker MUST live in its own importable .py file (tile_worker.py,
# next to this script) rather than being defined here. ProcessPoolExecutor
# on Windows spawns fresh interpreters that import-by-name whatever module
# defines the target function. That works for a normal script but breaks
# for functions defined inside a Jupyter/IPython notebook cell (including
# the QGIS Python console) -- there, the function's "module" is __main__,
# and a spawned worker's __main__ is its own launcher, not your notebook.
# The worker then dies on startup and every future raises BrokenProcessPool.
# If the import fails (e.g. tile_worker.py wasn't saved alongside this
# file), parallel processing is disabled automatically and the pipeline
# still runs correctly, just single-threaded.
try:
    from tile_worker import (
        process_tile_worker as _process_tile_worker,
        _preprocess_tile,
        _ridge_response_tile,
    )
    _WORKER_IMPORT_OK = True
except ImportError as _e:
    _process_tile_worker = None
    _WORKER_IMPORT_OK = False
    print(f"WARNING: could not import tile_worker.py ({_e}). "
          f"Make sure tile_worker.py is saved in the same folder as this script / "
          f"on sys.path. Falling back to sequential tile processing.")

    # Sequential-only fallbacks so the class still works with tile_worker.py
    # missing -- just without process-pool parallelism.
    def _preprocess_tile(band_u8, denoise_h, template_window, search_window, clahe_clip, clahe_tile):
        denoised = cv2.fastNlMeansDenoising(band_u8, h=denoise_h, templateWindowSize=template_window,
                                             searchWindowSize=search_window)
        clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=tuple(clahe_tile))
        return clahe.apply(denoised)

    def _ridge_response_tile(enhanced_u8, sigmas_px, alpha, beta, gamma):
        from skimage.filters import frangi
        img_f = enhanced_u8.astype(np.float32) / 255.0
        bright = frangi(img_f, sigmas=sigmas_px, alpha=alpha, beta=beta, gamma=gamma, black_ridges=False)
        dark = frangi(img_f, sigmas=sigmas_px, alpha=alpha, beta=beta, gamma=gamma, black_ridges=True)
        return np.maximum(bright, dark)

warnings.filterwarnings('ignore')


DEFAULT_CONFIG = {
    # --- tiling ---
    'tile_size': 1024,           # pixels, full resolution (no downsampling)
    'overlap_px': None,          # None = auto-derive from feature-width settings (recommended)
    'min_overlap_px': 64,        # floor applied even when auto-derived overlap would be smaller

    # --- image resolution ---
    'downsample_factor': 8,   # Optional: 2, 4, 8, etc. for faster processing. None = no downsampling
    'use_parallel_processing': True,  # Enable parallel tile processing
    'max_workers': None,         # Number of worker processes (None = auto)

    # --- ridge enhancement (Frangi/vesselness) ---
    'min_feature_width_m': 2.5,
    'ridge_max_feature_width_m': 40,   # ceiling for the ridge path; wider goes through wide-water path
    'frangi_num_scales': 4,            # sigmas are log-spaced between min/max width, derived per image
    'frangi_alpha': 0.5,
    'frangi_beta': 0.5,
    'frangi_gamma': 15,
    'ridge_response_percentile': 96,   # top N% of ridge response kept per tile

    # --- denoise / contrast (ridge path preprocessing) ---
    'denoise_h': 7,
    'denoise_template_window': 7,
    'denoise_search_window': 21,
    'clahe_clip_limit': 3.0,
    'clahe_tile_grid': (8, 8),

    # --- wide water detection (rivers/canals beyond the ridge ceiling) ---
    'wide_water_min_width_m': 30,      # deliberately overlaps ridge_max_feature_width_m so nothing falls in a gap
    'wide_water_box_size_m': 15,       # local-statistics window for darkness/uniformity test
    'wide_water_dark_percentile': 35,  # pixels darker than this percentile (per-image) are water candidates
    'wide_water_max_local_std': 12,    # low local std = visually uniform, water-like
    'wide_water_min_elongation_ratio': 1.5,  # relaxed vs. ridge path -- wide rivers are less elongated within one tile

    # --- morphology / component filtering (ridge path) ---
    'closing_radius_px': 2,
    'min_component_length_m': 20,
    'min_elongation_ratio': 3.0,
    'max_solidity': 0.65,

    # --- type heuristic thresholds (rough starting points, meters) ---
    'railway_width_range_m': (3, 8),
    'road_width_range_m': (3, 15),
    'river_width_range_m': (4, 200),
    'railway_max_sinuosity': 1.05,
    'river_min_sinuosity': 1.15,

    # --- OSM refinement ---
    'osm_match_buffer_m': 15,
}


# NOTE: the tile worker functions (_preprocess_tile, _ridge_response_tile,
# process_tile_worker) now live in tile_worker.py and are imported above as
# _process_tile_worker. See the comment at the top of this file for why.


class GeorefLinearFeaturePipelineOptimized:
    """
    Optimized pipeline for extracting linear features (roads, rivers, railways)
    with performance enhancements including optional downsampling and genuine
    parallel tile processing.
    """

    def __init__(self, source_path, output_dir, config=None):
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.config = {**DEFAULT_CONFIG, **(config or {})}

        self.csv_dir = self.output_dir / 'csv_outputs'
        self.vector_dir = self.output_dir / 'vector_outputs'
        self.metadata_dir = self.output_dir / 'metadata'
        for d in [self.csv_dir, self.vector_dir, self.metadata_dir]:
            d.mkdir(parents=True, exist_ok=True)

        self.shortcuts = []
        self.shortcuts_with_targets = []
        self.raster_data = []
        self.all_images = []
        self.image_data_df = None

        print(f"Source path: {self.source_path}")
        print(f"Output directory: {self.output_dir}")

        # Lock for thread-safe progress tracking (main-process bookkeeping only;
        # never passed into worker processes)
        self.progress_lock = Lock()
        self.processed_tiles = 0
        self.total_tiles = 0

        # Persistent worker pool, created lazily on first use and reused across
        # every image in the run (spawning processes repeatedly is expensive).
        self._pool = None

        # Flipped to True the first time the process pool breaks (e.g. when
        # running from a notebook/console where spawned workers can't import
        # the worker function) or tile_worker.py couldn't be imported at all.
        # Once True, we stop retrying parallel mode for the rest of the run
        # and just use the sequential path -- still correct, just slower.
        self._parallel_disabled = not _WORKER_IMPORT_OK

    # ==================== POOL LIFECYCLE ====================

    def _ensure_pool(self):
        if self._pool is None:
            self._pool = ProcessPoolExecutor(max_workers=self.config['max_workers'])
        return self._pool

    def close(self):
        if self._pool is not None:
            self._pool.shutdown(wait=True)
            self._pool = None

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass

    # ==================== GEOREFERENCING (unchanged logic) ====================

    def parse_image_info(self, filename):
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').replace('_Finalized', '')
        parts = [p.strip() for p in filename.split('#') if p.strip()]
        image_num = flight_num = sortie = bestell_id = date = None
        if parts:
            image_num = parts[0]
        for part in parts:
            if part.startswith('ID') and any(c.isdigit() for c in part[2:]):
                numbers = re.findall(r'\d+', part)
                bestell_id = numbers[0] if numbers else None
                break
        for part in parts:
            if re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                date = part
                break
        if len(parts) >= 3:
            for part in parts[1:-1]:
                if part != bestell_id and not re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                    if not flight_num:
                        flight_num = part
                    elif not sortie:
                        sortie = part
        return {'image_number': image_num, 'flight_number': flight_num, 'sortie': sortie,
                'bestell_id': bestell_id, 'date': date, 'original_name': filename}

    def find_and_process_shortcuts(self):
        print(f"\nScanning for shortcut files in {self.source_path}...")
        for root, _, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    info = self.parse_image_info(file[:-4])
                    info['full_path'] = Path(root) / file
                    info['folder'] = root
                    self.shortcuts.append(info)
        print(f"Found {len(self.shortcuts)} shortcut files")
        return self.shortcuts

    def get_shortcut_target(self, lnk_path):
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            target = shell.CreateShortCut(str(lnk_path)).TargetPath
            pythoncom.CoUninitialize()
            return target if target else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path}: {e}")
            return None

    def find_modified_versions(self, original_path):
        if not original_path or not original_path.exists():
            return []
        folder = original_path.parent
        stem = original_path.stem
        patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        found = []
        for pattern in patterns:
            candidate = folder / f"{stem}{pattern}{original_path.suffix}"
            if candidate.exists():
                found.append(candidate)
        for f in folder.iterdir():
            if f.is_file() and f.suffix == original_path.suffix:
                if f.stem.startswith(stem) and any(p in f.stem for p in patterns):
                    found.append(f)
        return list(set(found))

    def extract_shortcut_targets(self):
        print("\nExtracting shortcut targets and finding modified versions...")
        for sc in self.shortcuts:
            target = self.get_shortcut_target(sc['full_path'])
            if not target:
                continue
            target = Path(target)
            sc.update({'target_path': target, 'target_exists': target.exists(),
                       'target_folder': target.parent, 'target_filename': target.name})
            info = self.parse_image_info(target.stem)
            sc.update({'target_image_number': info['image_number'], 'target_bestell_id': info['bestell_id'],
                       'target_sortie': info['sortie'], 'target_date': info['date']})
            if sc['target_exists']:
                mods = self.find_modified_versions(target)
                if mods:
                    sc['modified_files'] = mods
            self.shortcuts_with_targets.append(sc)
        print(f"Successfully processed {len(self.shortcuts_with_targets)} shortcuts")
        return self.shortcuts_with_targets

    def calculate_tiff_area_extent(self, tiff_path):
        try:
            with rasterio.open(tiff_path) as src:
                transform = src.transform
                pixel_res_x, pixel_res_y = transform.a, -transform.e
                bounds = src.bounds
                return {
                    'rows': int(src.height), 'cols': int(src.width),
                    'pixel_res_x_m': float(pixel_res_x), 'pixel_res_y_m': float(pixel_res_y),
                    'bounds': bounds, 'is_georeferenced': src.crs is not None,
                    'crs': src.crs, 'transform': transform,
                }
        except Exception as e:
            print(f"Error reading TIFF {tiff_path.name}: {e}")
            return None

    def process_georeferenced_images(self):
        print("\nProcessing georeferenced TIFF files...")
        for sc in self.shortcuts_with_targets:
            bestell_id = sc.get('bestell_id') or sc.get('target_bestell_id')
            bid = float(bestell_id) if bestell_id and bestell_id.isdigit() else 0.0
            base = {
                'BestellID': bid, 'SORTIE': sc.get('sortie') or sc.get('target_sortie') or '',
                'Bildnummer': sc.get('image_number') or sc.get('target_image_number') or '',
                'Has_Georef': False, 'Image_Name': sc.get('target_filename', ''),
                'Image_Path': str(sc.get('target_path', '')),
            }
            has_georef = False
            for mf in sc.get('modified_files', []):
                if mf.suffix.lower() in ['.tif', '.tiff']:
                    info = self.calculate_tiff_area_extent(mf)
                    if info and info['is_georeferenced']:
                        has_georef = True
                        rec = base.copy()
                        rec.update({
                            'Rows': float(info['rows']), 'Cols': float(info['cols']),
                            'Res_m': float((info['pixel_res_x_m'] + info['pixel_res_y_m']) / 2),
                            'Has_Georef': True, 'Image_Name': mf.name, 'Image_Path': str(mf),
                            'crs': info['crs'], 'transform': info['transform'],
                        })
                        self.raster_data.append(rec)
                        self.all_images.append(rec)
            if not has_georef:
                self.all_images.append(base)

        self.image_data_df = pd.DataFrame(self.raster_data) if self.raster_data else pd.DataFrame()
        print(f"Found {len(self.raster_data)} georeferenced images")
        return self.raster_data, self.all_images

    def save_georef_csv(self):
        if not self.all_images:
            return None
        path = self.csv_dir / "Raster_ImageExtent.csv"
        rows = [{k: v for k, v in r.items() if k not in ['Image_Path', 'crs', 'transform']}
                for r in self.all_images]
        fieldnames = sorted({k for r in rows for k in r.keys()})
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"Georeferencing CSV saved to: {path}")
        return path

    # ==================== SHARED IMAGE PREP ====================

    def _preprocess(self, band_u8):
        """Sequential-path equivalent of the module-level worker preprocessing
        (used only when parallel processing is off or there are too few tiles
        to bother spawning workers)."""
        cfg = self.config
        return _preprocess_tile(
            band_u8, cfg['denoise_h'], cfg['denoise_template_window'],
            cfg['denoise_search_window'], cfg['clahe_clip_limit'], cfg['clahe_tile_grid'],
        )

    def _normalize_band(self, band):
        band = np.nan_to_num(band, nan=0, posinf=255, neginf=0)
        if band.dtype != np.uint8:
            if band.max() > band.min():
                band = ((band - band.min()) / (band.max() - band.min()) * 255).astype(np.uint8)
            else:
                band = np.zeros_like(band, dtype=np.uint8)
        return band

    def _resolve_scale_params(self, pixel_res_m):
        """
        Derive Frangi sigmas (pixels), tile overlap (pixels), and the
        wide-water box filter size (pixels) from this image's actual
        pixel resolution and the configured feature-width settings
        (meters). This replaces hardcoded pixel constants, which only
        make sense at one specific resolution and silently mismatch on
        images at a different scale or when width settings change.
        """
        cfg = self.config

        min_w_px = max(cfg['min_feature_width_m'] / pixel_res_m, 1.0)
        max_w_px = max(cfg['ridge_max_feature_width_m'] / pixel_res_m, min_w_px + 1)

        # Frangi sigma is roughly half the ridge width. Log-spacing covers
        # a wide width range with far fewer scales than linear spacing
        # would need (cheaper, and each added scale multiplies runtime).
        sigma_lo = max(min_w_px / 2, 1.0)
        sigma_hi = max(max_w_px / 2, sigma_lo + 1)
        sigmas_px = sorted(set(
            int(s) for s in np.round(np.geomspace(sigma_lo, sigma_hi, cfg['frangi_num_scales']))
            if s >= 1
        ))
        if not sigmas_px:
            sigmas_px = [1]

        # Overlap must exceed the widest feature the ridge path targets
        # (with margin) so a feature isn't clipped right at a tile edge.
        auto_overlap = int(max_w_px * 1.5)
        overlap_px = cfg['overlap_px'] if cfg['overlap_px'] is not None else auto_overlap
        overlap_px = max(overlap_px, cfg['min_overlap_px'])

        wide_water_box_px = max(int(cfg['wide_water_box_size_m'] / pixel_res_m), 3)
        if wide_water_box_px % 2 == 0:
            wide_water_box_px += 1  # odd size for a centered box filter

        return {'sigmas_px': sigmas_px, 'overlap_px': overlap_px, 'wide_water_box_px': wide_water_box_px}

    # ==================== RIDGE PATH (roads, rail, narrow/medium rivers) ====================

    def _ridge_response(self, enhanced_u8, sigmas_px):
        """Sequential-path equivalent of the module-level worker ridge response."""
        cfg = self.config
        return _ridge_response_tile(enhanced_u8, sigmas_px, cfg['frangi_alpha'], cfg['frangi_beta'], cfg['frangi_gamma'])

    def _tile_windows(self, height, width, overlap_px):
        ts = self.config['tile_size']
        step = max(ts - overlap_px, 1)
        r0 = 0
        while r0 < height:
            r1 = min(r0 + ts, height)
            c0 = 0
            while c0 < width:
                c1 = min(c0 + ts, width)
                yield r0, r1, c0, c1
                if c1 == width:
                    break
                c0 += step
            if r1 == height:
                break
            r0 += step

    def build_full_resolution_masks(self, image_path):
        """
        Reads the full-resolution band once, then builds TWO full-size
        boolean masks by tiling and stitching via logical OR:
          - ridge_mask: Frangi response, for roads/rail/narrow-medium rivers
          - wide_water_mask: darkness+uniformity, for broad rivers/canals
        Both use resolution-derived parameters (see _resolve_scale_params).

        Note on memory: this keeps a couple of boolean arrays the size of
        the source image in memory. For very large scans (>8000x8000)
        consider a per-tile-polygonize-then-union variant instead.
        """
        with rasterio.open(image_path) as src:
            height, width = src.height, src.width
            transform = src.transform
            crs = src.crs
            pixel_res_m = abs((transform.a + -transform.e) / 2)
            band = self._normalize_band(src.read(1))

        # Optional: Downsample for faster processing
        downsample_factor = self.config['downsample_factor']
        if downsample_factor and downsample_factor > 1:
            print(f"  Downsampling image by factor {downsample_factor} for faster processing...")
            new_height = max(1, height // downsample_factor)
            new_width = max(1, width // downsample_factor)
            band = cv2.resize(band, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
            height, width = new_height, new_width
            transform = transform * Affine.scale(downsample_factor, downsample_factor)
            print(f"  Downsampled image size: {width}x{height}")

        if height * width > 8000 * 8000:
            print(f"  Warning: large image ({width}x{height}); full-mask memory use may be significant.")

        # Ensure contiguous uint8 buffer so it can be dropped straight into shared memory
        band = np.ascontiguousarray(band, dtype=np.uint8)

        scale_params = self._resolve_scale_params(pixel_res_m)
        sigmas_px, overlap_px = scale_params['sigmas_px'], scale_params['overlap_px']
        print(f"  pixel_res={pixel_res_m:.3f} m -> frangi sigmas(px)={sigmas_px}, tile overlap(px)={overlap_px}")

        ridge_mask = np.zeros((height, width), dtype=bool)
        pct = self.config['ridge_response_percentile']
        cfg = self.config

        tiles = list(self._tile_windows(height, width, overlap_px))
        self.total_tiles = len(tiles)

        ridge_params = {
            'denoise_h': cfg['denoise_h'],
            'template_window': cfg['denoise_template_window'],
            'search_window': cfg['denoise_search_window'],
            'clahe_clip': cfg['clahe_clip_limit'],
            'clahe_tile': cfg['clahe_tile_grid'],
            'alpha': cfg['frangi_alpha'],
            'beta': cfg['frangi_beta'],
            'gamma': cfg['frangi_gamma'],
            'percentile': pct,
        }

        use_parallel = cfg['use_parallel_processing'] and not self._parallel_disabled and len(tiles) > 4
        parallel_failed = False

        if use_parallel:
            print(f"  Processing {len(tiles)} tiles in parallel...")
            shm = shared_memory.SharedMemory(create=True, size=band.nbytes)
            try:
                shm_band = np.ndarray(band.shape, dtype=band.dtype, buffer=shm.buf)
                shm_band[:] = band[:]

                pool = self._ensure_pool()
                futures = [
                    pool.submit(_process_tile_worker, shm.name, band.shape, str(band.dtype),
                                (r0, r1, c0, c1), sigmas_px, ridge_params)
                    for (r0, r1, c0, c1) in tiles
                ]

                completed = 0
                for future in as_completed(futures):
                    r0, r1, c0, c1, tile_mask = future.result()
                    ridge_mask[r0:r1, c0:c1] |= tile_mask
                    completed += 1
                    if completed % 10 == 0 or completed == len(futures):
                        print(f"  Processed {completed}/{len(futures)} tiles...")

            except BrokenProcessPool as e:
                # Most common cause: this pipeline is running from a notebook /
                # interactive console (e.g. QGIS Python console) on Windows,
                # where spawned worker processes can't import a function
                # defined outside a real, importable module. Discard the dead
                # pool, stop retrying parallel mode for the rest of the run,
                # and fall through to the sequential path below for this image.
                print(f"  WARNING: worker pool broke ({e}). Disabling parallel "
                      f"processing for the rest of this run and falling back "
                      f"to sequential tile processing.")
                self.close()  # drop the broken pool so a stale one is never reused
                self._parallel_disabled = True
                parallel_failed = True
            finally:
                shm.close()
                shm.unlink()

        if not use_parallel or parallel_failed:
            # Sequential fallback (small tile counts, parallel disabled, or
            # parallel mode just broke for this image). Note ridge_mask may
            # already contain results from tiles that completed before a
            # mid-run failure; OR-ing in the same tiles again is harmless.
            cfg2 = self.config
            for r0, r1, c0, c1 in tqdm(tiles, desc=f"  Ridge-filtering {Path(image_path).name}", leave=False):
                tile = band[r0:r1, c0:c1]
                if tile.shape[0] < 16 or tile.shape[1] < 16:
                    continue
                enhanced = _preprocess_tile(
                    tile, cfg2['denoise_h'], cfg2['denoise_template_window'],
                    cfg2['denoise_search_window'], cfg2['clahe_clip_limit'], cfg2['clahe_tile_grid'],
                )
                response = _ridge_response_tile(
                    enhanced, sigmas_px, cfg2['frangi_alpha'], cfg2['frangi_beta'], cfg2['frangi_gamma'],
                )
                if response.max() > response.min():
                    thresh = np.percentile(response, pct)
                    tile_mask = response >= thresh
                else:
                    tile_mask = np.zeros_like(response, dtype=bool)
                ridge_mask[r0:r1, c0:c1] |= tile_mask
                self.processed_tiles += 1

        # Wide-water detection (single pass over the full band, not tiled)
        wide_water_mask = self._detect_wide_water(band, scale_params['wide_water_box_px'])

        gc.collect()

        return {
            'ridge_mask': ridge_mask,
            'wide_water_mask': wide_water_mask,
            'band': band,
            'transform': transform,
            'crs': crs,
            'pixel_res_m': pixel_res_m,
        }

    # ==================== WIDE-WATER PATH ====================

    def _detect_wide_water(self, band_u8, box_px):
        """
        Broad rivers/canals above the ridge filter's practical width
        ceiling: flagged by local darkness + local uniformity (low local
        std), computed with a box filter. Water in panchromatic WWII
        imagery tends to read darker and texturally flatter than built-up
        or vegetated land -- a much better fit than a ridge filter for
        features that are wide relative to a tile.

        Uses cv2.boxFilter / cv2.sqrBoxFilter, which are integral-image
        based (cost independent of window size), instead of an explicit
        convolution with a manually built ones-kernel.
        """
        img_f = band_u8.astype(np.float32)

        local_mean = cv2.boxFilter(img_f, -1, (box_px, box_px), borderType=cv2.BORDER_REFLECT)
        local_sqmean = cv2.sqrBoxFilter(img_f, -1, (box_px, box_px), borderType=cv2.BORDER_REFLECT)
        local_std = np.sqrt(np.clip(local_sqmean - local_mean ** 2, 0, None))

        dark_thresh = np.percentile(local_mean, self.config['wide_water_dark_percentile'])
        mask = (local_mean <= dark_thresh) & (local_std <= self.config['wide_water_max_local_std'])
        return mask

    # ==================== COMPONENT METRICS / CLASSIFICATION (shared) ====================

    def clean_and_label(self, mask, pixel_res_m):
        closing_r = self.config['closing_radius_px']
        closed = binary_closing(mask, disk(closing_r))
        min_len_px = max(int(self.config['min_component_length_m'] / pixel_res_m), 3)
        cleaned = remove_small_objects(closed, min_size=min_len_px * 2)
        return label(cleaned, connectivity=2)

    def _component_metrics(self, comp_mask, pixel_res_m):
        """Width via distance transform along the skeleton, length via
        skeleton pixel count, sinuosity via skeleton path length vs.
        straight-line distance between the two most distant skeleton
        pixels. Approximations -- true centerline/length for branching
        networks would need graph-based skeleton analysis -- but enough
        for a first-pass type guess and OSM comparison metrics."""
        skel = skeletonize(comp_mask)
        skel_coords = np.column_stack(np.nonzero(skel))
        if len(skel_coords) < 2:
            return None

        dist = ndi.distance_transform_edt(comp_mask)
        widths_px = dist[tuple(skel_coords.T)]
        mean_width_m = float(np.mean(widths_px) * 2 * pixel_res_m)

        length_px = len(skel_coords)
        length_m = float(length_px * pixel_res_m)

        sample = (skel_coords if len(skel_coords) <= 500
                  else skel_coords[np.random.choice(len(skel_coords), 500, replace=False)])
        d = np.linalg.norm(sample[:, None, :] - sample[None, :, :], axis=-1)
        i, j = np.unravel_index(np.argmax(d), d.shape)
        straight_px = d[i, j]
        straight_m = float(straight_px * pixel_res_m) if straight_px > 0 else 1e-6
        sinuosity = max(float(length_m / straight_m) if straight_m > 0 else 1.0, 1.0)

        region = regionprops(comp_mask.astype(np.uint8))[0]
        elongation = (region.major_axis_length / region.minor_axis_length
                      if region.minor_axis_length > 0 else np.inf)

        return {
            'mean_width_m': mean_width_m, 'length_m': length_m, 'sinuosity': sinuosity,
            'solidity': float(region.solidity), 'elongation': float(elongation),
            'area_px': int(region.area),
        }

    def classify_linear_type(self, metrics, mean_intensity):
        """
        Rough heuristic type guess from width/straightness/intensity --
        a fallback only. Once compared against OSM, prefer
        `refine_classes_with_osm`, which assigns the real tag from the
        nearest matching OSM feature; pixel heuristics alone can't
        reliably separate e.g. a farm track from a minor road.
        """
        cfg = self.config
        w, sinuosity = metrics['mean_width_m'], metrics['sinuosity']
        rw_lo, rw_hi = cfg['railway_width_range_m']
        rd_lo, rd_hi = cfg['road_width_range_m']
        rv_lo, rv_hi = cfg['river_width_range_m']

        if rw_lo <= w <= rw_hi and sinuosity <= cfg['railway_max_sinuosity']:
            return 'probable_railway', 0.5
        if w >= rv_lo and sinuosity >= cfg['river_min_sinuosity'] and mean_intensity < 110:
            return 'probable_river', 0.5
        if rd_lo <= w <= rd_hi:
            return 'probable_road', 0.4
        return 'linear_feature_unclassified', 0.2

    def polygonize_component(self, comp_mask, r_offset, c_offset, full_shape, transform):
        full_mask = np.zeros(full_shape, dtype=np.uint8)
        full_mask[r_offset:r_offset + comp_mask.shape[0],
                  c_offset:c_offset + comp_mask.shape[1]] = comp_mask.astype(np.uint8)
        polys = [shape(geom) for geom, val in rio_features.shapes(
            full_mask, mask=full_mask.astype(bool), transform=transform) if val == 1]
        if not polys:
            return None
        return unary_union(polys).simplify(max(transform.a, 1e-6))

    def _components_to_records(self, mask, pixel_res_m, full_band, transform, image_row, crs,
                                min_elongation, max_solidity, forced_type=None, detection_method='ridge'):
        labeled = self.clean_and_label(mask, pixel_res_m)
        records = []
        for region in regionprops(labeled):
            minr, minc, maxr, maxc = region.bbox
            comp_mask = labeled[minr:maxr, minc:maxc] == region.label

            metrics = self._component_metrics(comp_mask, pixel_res_m)
            if metrics is None:
                continue
            if metrics['length_m'] < self.config['min_component_length_m']:
                continue
            if metrics['elongation'] < min_elongation:
                continue
            if metrics['solidity'] > max_solidity:
                continue

            mean_intensity = float(np.mean(full_band[minr:maxr, minc:maxc][comp_mask]))
            if forced_type:
                feature_type, confidence = forced_type, 0.4
            else:
                feature_type, confidence = self.classify_linear_type(metrics, mean_intensity)

            polygon = self.polygonize_component(comp_mask, minr, minc, labeled.shape, transform)
            if polygon is None or polygon.is_empty:
                continue

            records.append({
                'image_name': Path(image_row['Image_Path']).name,
                'image_path': image_row['Image_Path'],
                'component_id': f"{detection_method}_{int(region.label)}",
                'detection_method': detection_method,
                'bestell_id': image_row.get('BestellID', 0),
                'sortie': image_row.get('SORTIE', ''),
                'bildnummer': image_row.get('Bildnummer', ''),
                'feature_type_guess': feature_type,
                'type_confidence': confidence,
                'mean_width_m': metrics['mean_width_m'],
                'length_m': metrics['length_m'],
                'sinuosity': metrics['sinuosity'],
                'elongation': metrics['elongation'],
                'solidity': metrics['solidity'],
                'mean_intensity': mean_intensity,
                'geometry': polygon,
            })
        return records

    # ==================== PER-IMAGE ORCHESTRATION ====================

    def extract_linear_features(self, image_row):
        image_path = Path(image_row['Image_Path'])
        if not image_path.exists():
            print(f"Image not found: {image_path}")
            return None

        print(f"\nProcessing: {image_path.name}")
        built = self.build_full_resolution_masks(image_path)

        ridge_records = self._components_to_records(
            built['ridge_mask'], built['pixel_res_m'], built['band'], built['transform'],
            image_row, built['crs'],
            min_elongation=self.config['min_elongation_ratio'],
            max_solidity=self.config['max_solidity'],
            forced_type=None, detection_method='ridge',
        )

        water_records = self._components_to_records(
            built['wide_water_mask'], built['pixel_res_m'], built['band'], built['transform'],
            image_row, built['crs'],
            min_elongation=self.config['wide_water_min_elongation_ratio'],
            max_solidity=0.95,  # wide water bodies are naturally more solid/blob-like than roads
            forced_type='probable_river_wide', detection_method='wide_water',
        )

        records = ridge_records + water_records
        if not records:
            print("  No linear features passed filtering.")
            return None

        gdf = gpd.GeoDataFrame(
            records,
            geometry='geometry',
            crs=built['crs']
        )
        print(f"  Found {len(ridge_records)} ridge-path + {len(water_records)} wide-water candidates.")
        return gdf

    # ==================== OPTIONAL OSM REFINEMENT ====================

    def refine_classes_with_osm(self, gdf, osm_path, type_field='fclass', buffer_m=None):
        """
        Nearest-neighbor join against a local OSM extract (e.g. a
        .gpkg/.shp with roads+railways+waterways merged, with a column
        identifying feature type). Candidates within `buffer_m` of an OSM
        feature get the OSM tag (confidence 1.0), overriding the pixel
        heuristic. Unmatched candidates keep their heuristic guess --
        exactly the interesting cases for historical comparison (features
        present in 1944 imagery with no current OSM counterpart, or vice
        versa). Requires geopandas >= 0.10 (sjoin_nearest).
        """
        if buffer_m is None:
            buffer_m = self.config['osm_match_buffer_m']
        osm = gpd.read_file(osm_path)
        if osm.crs != gdf.crs:
            osm = osm.to_crs(gdf.crs)

        joined = gpd.sjoin_nearest(gdf, osm[[type_field, 'geometry']],
                                    max_distance=buffer_m, distance_col='osm_dist_m')
        joined['feature_type_final'] = joined[type_field].fillna(joined['feature_type_guess'])
        joined['type_confidence'] = np.where(joined[type_field].notna(), 1.0, joined['type_confidence'])
        return joined.drop(columns=[type_field])

    # ==================== SAVE / TOP-LEVEL RUN ====================

    def save_results(self, image_name, gdf):
        stem = Path(image_name).stem
        gdf.to_file(self.vector_dir / f"{stem}_linear_features.gpkg", driver='GPKG')
        gdf.to_file(self.vector_dir / f"{stem}_linear_features.geojson", driver='GeoJSON')
        meta = {
            'image_name': image_name, 'num_features': len(gdf),
            'type_counts': gdf['feature_type_guess'].value_counts().to_dict(),
            'total_length_m': float(gdf['length_m'].sum()),
        }
        with open(self.metadata_dir / f"{stem}_metadata.json", 'w') as f:
            json.dump(meta, f, indent=2, default=str)

    def process_all_images(self, max_images=None, osm_path=None, osm_type_field='fclass'):
        if self.image_data_df is None or len(self.image_data_df) == 0:
            print("No georeferenced images to process")
            return []

        images = self.image_data_df.head(max_images) if max_images else self.image_data_df
        all_gdfs = []
        for _, row in images.iterrows():
            gdf = self.extract_linear_features(row)
            if gdf is None:
                continue
            if osm_path:
                gdf = self.refine_classes_with_osm(gdf, osm_path, type_field=osm_type_field)
            self.save_results(row['Image_Name'], gdf)
            all_gdfs.append(gdf)

        if all_gdfs:
            combined = gpd.GeoDataFrame(pd.concat(all_gdfs, ignore_index=True))
            combined_path = self.output_dir / 'all_linear_features.gpkg'
            combined.to_file(combined_path, driver='GPKG')
            print(f"\nCombined output saved to: {combined_path}")
        return all_gdfs

    def run(self, max_images=None, osm_path=None, osm_type_field='fclass'):
        print("\n" + "=" * 80)
        print("GEOREFERENCE + LINEAR FEATURE EXTRACTION PIPELINE (OPTIMIZED)")
        print("=" * 80)

        try:
            self.find_and_process_shortcuts()
            self.extract_shortcut_targets()
            self.process_georeferenced_images()
            csv_path = self.save_georef_csv()
            print(f"\nGeoreferenced images found: {len(self.raster_data)}")

            results = self.process_all_images(max_images=max_images, osm_path=osm_path,
                                               osm_type_field=osm_type_field)
        finally:
            # Always tear down the worker pool, even if processing raised.
            self.close()

        print("\n" + "=" * 80)
        print("PIPELINE COMPLETE")
        print("=" * 80)
        print(f"Output directory: {self.output_dir}")
        print(f"Georeferencing CSV: {csv_path}")
        print(f"Images with extracted features: {len(results)}")

        return {
            'georeferencing': {'shortcuts_count': len(self.shortcuts),
                                'georef_images_count': len(self.raster_data),
                                'csv_path': str(csv_path) if csv_path else None},
            'linear_features': {'images_processed': len(results)},
        }


if __name__ == "__main__":
    SOURCE_PATH = r"W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\2. FINALIZED\1. HarmonyMarked_Finalized\ID1732 Stuttgart-Mühlhausen - LBDB"
    OUTPUT_DIR = r"D:\FBM try\outputs"

    # Point at a local OSM extract (roads+rail+waterways merged, with a
    # type column) for automatic class refinement during the run. Leave
    # as None to only get the pixel-heuristic guess and do the OSM
    # comparison as a separate step afterward.
    OSM_REFERENCE_PATH = None   # e.g. r"D:\FBM try\osm\stuttgart_transport_water.gpkg"
    OSM_TYPE_FIELD = 'fclass'

    pipeline = GeorefLinearFeaturePipelineOptimized(SOURCE_PATH, OUTPUT_DIR, config=dict(DEFAULT_CONFIG))
    results = pipeline.run(max_images=None, osm_path=OSM_REFERENCE_PATH, osm_type_field=OSM_TYPE_FIELD)

    print(f"\nPipeline completed. Results saved to: {OUTPUT_DIR}")

### Morphological Filtering : Canny edge detection + OSM road layer cross-check

In [ ]:
import os
import sys
from pathlib import Path

# ---------------------------------------------------------
# PROJ and GDAL environment - MUST BE SET BEFORE IMPORTS
# ---------------------------------------------------------

QGIS_ROOT = Path(r"C:\Program Files\QGIS 3.34.12")

PROJ_DIR = QGIS_ROOT / "share" / "proj"
GDAL_DIR = QGIS_ROOT / "share" / "gdal"

# Set environment variables BEFORE any imports
os.environ["PROJ_DATA"] = str(PROJ_DIR)
os.environ["PROJ_LIB"] = str(PROJ_DIR)  # compatibility
os.environ["GDAL_DATA"] = str(GDAL_DIR)

# Add QGIS Python path to system path
os.environ["PATH"] = (
    str(QGIS_ROOT / "bin")
    + os.pathsep
    + str(QGIS_ROOT / "apps" / "Python312")
    + os.pathsep
    + str(QGIS_ROOT / "apps" / "Python312" / "Scripts")
    + os.pathsep
    + os.environ.get("PATH", "")
)

# Also set for PROJ directly
os.environ["PROJ_DIR"] = str(PROJ_DIR)

print("QGIS_ROOT:", QGIS_ROOT)
print("PROJ_DIR:", PROJ_DIR)
print("GDAL_DIR:", GDAL_DIR)
print("proj.db exists:", (PROJ_DIR / "proj.db").exists())

# ---------------------------------------------------------
# Now import libraries
# ---------------------------------------------------------

import psycopg2
from psycopg2 import sql
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import box, Polygon
from shapely.ops import clip_by_rect

# Set PROJ path again for pyproj
try:
    import pyproj
    pyproj.datadir.set_data_dir(str(PROJ_DIR))
    print(f"pyproj data dir set to: {pyproj.datadir.get_data_dir()}")
except ImportError:
    print("pyproj not available, continuing...")

# Force GDAL to use the correct PROJ
try:
    from osgeo import gdal, ogr, osr
    # Set PROJ through GDAL
    gdal.SetConfigOption('PROJ_DATA', str(PROJ_DIR))
    gdal.SetConfigOption('PROJ_LIB', str(PROJ_DIR))
    print("GDAL PROJ config set")
except:
    print("GDAL not yet imported, will set later")

# Now import the rest
import cv2
import matplotlib.pyplot as plt
from shapely.geometry import LineString, box, Polygon, MultiLineString
import osmnx as ox
import warnings
import re
import win32com.client
import pythoncom
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# Database configuration
# ---------------------------------------------------------

DB_CONFIG = {
    'host': '192.168.190.10',
    'port': 5432,
    'database': 'Server_LBA',
    'user': 'LBA_team',
    'password': 'lba',
    'sslmode': 'prefer'
}

# ---------------------------------------------------------
# Set PROJ path for all libraries
# ---------------------------------------------------------

PROJ_PATH = r'C:\Program Files\QGIS 3.34.12\share\proj'
os.environ['PROJ_DATA'] = PROJ_PATH
os.environ['PROJ_LIB'] = PROJ_PATH

# For pyproj
try:
    import pyproj
    pyproj.datadir.set_data_dir(PROJ_PATH)
except:
    pass

# For GDAL
try:
    from osgeo import gdal
    gdal.SetConfigOption('PROJ_DATA', PROJ_PATH)
    gdal.SetConfigOption('PROJ_LIB', PROJ_PATH)
except:
    pass

class GeorefEdgeProcessor:
    def __init__(self, source_path, output_dir):
        self.source_path = Path(source_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        self.shortcuts = []
        self.shortcuts_with_targets = []
        
        self.db_connection = None
        
        # Buffer distances (will be set once)
        self.buffer_edges = None
        self.buffer_osm = None
        
        # Results storage
        self.all_results = []
        
        print(f"Initialized GeorefEdgeProcessor")
        print(f"Source path: {self.source_path}")
        print(f"Output directory: {self.output_dir}")
        
        # Verify PROJ is working
        self.verify_proj()

    def verify_proj(self):
        """Verify that PROJ is correctly configured."""
        try:
            import pyproj
            crs = pyproj.CRS("EPSG:4326")
            print(f"✓ PROJ working: EPSG:4326 is valid")
            return True
        except Exception as e:
            print(f"✗ PROJ issue: {e}")
            print("  Trying to fix PROJ path...")
            try:
                import pyproj
                pyproj.datadir.set_data_dir(r'C:\Program Files\QGIS 3.34.12\share\proj')
                crs = pyproj.CRS("EPSG:4326")
                print(f"✓ PROJ fixed: EPSG:4326 is valid")
                return True
            except:
                print("  Could not fix PROJ automatically")
                return False

    def get_buffer_distances(self):
        """Get buffer distances once for all images."""
        print("\n" + "=" * 80)
        print("BUFFER DISTANCE CONFIGURATION")
        print("=" * 80)
        print("These buffer distances will be applied to ALL images.")
        
        while True:
            try:
                self.buffer_edges = float(input("\nEnter buffer distance for edges (meters): "))
                break
            except ValueError:
                print("Invalid input. Please enter a number.")
        
        while True:
            try:
                self.buffer_osm = float(input("Enter buffer distance for OSM roads (meters): "))
                break
            except ValueError:
                print("Invalid input. Please enter a number.")
        
        print(f"\nBuffer distances set:")
        print(f"  Edge buffer: {self.buffer_edges} meters")
        print(f"  OSM buffer: {self.buffer_osm} meters")
        print("=" * 80)

    def connect_database(self):
        try:
            self.db_connection = psycopg2.connect(
                host=DB_CONFIG['host'],
                port=DB_CONFIG['port'],
                database=DB_CONFIG['database'],
                user=DB_CONFIG['user'],
                password=DB_CONFIG['password'],
                sslmode=DB_CONFIG['sslmode']
            )
            print("Successfully connected to the database")
            return True
        except Exception as e:
            print(f"Error connecting to database: {e}")
            return False

    def close_database(self):
        if self.db_connection:
            self.db_connection.close()
            print("Database connection closed")

    def get_image_polygon(self, sortie, image_num):
        if not self.db_connection:
            print("No database connection available")
            return None
        
        try:
            cursor = self.db_connection.cursor()
            
            cursor.execute("""
                SELECT column_name 
                FROM information_schema.columns 
                WHERE table_name = 'bildumrahmungen_aktuell'
            """)
            columns = [col[0] for col in cursor.fetchall()]
            
            sortie_col = None
            bild_col = None
            geom_col = None
            
            for col in columns:
                col_lower = col.lower()
                if 'sortie' in col_lower:
                    sortie_col = col
                if 'bild' in col_lower and ('num' in col_lower or 'nr' in col_lower or 'nummer' in col_lower):
                    bild_col = col
                if 'geom' in col_lower:
                    geom_col = col
            
            if not sortie_col or not bild_col or not geom_col:
                print(f"Could not find required columns. Found: {columns}")
                return None
            
            print(f"Using columns: sortie='{sortie_col}', bild='{bild_col}', geom='{geom_col}'")
            
            query = sql.SQL("""
                SELECT 
                    {sortie_col},
                    {bild_col},
                    ST_AsText({geom_col}) as geometry_wkt,
                    ST_SRID({geom_col}) as srid
                FROM bildumrahmungen_aktuell
                WHERE {sortie_col} = %s AND {bild_col} = %s
            """).format(
                sortie_col=sql.Identifier(sortie_col),
                bild_col=sql.Identifier(bild_col),
                geom_col=sql.Identifier(geom_col)
            )
            
            print(f"Querying database with sortie='{sortie}', bildnummer='{image_num}'")
            cursor.execute(query, (sortie, image_num))
            result = cursor.fetchone()
            cursor.close()
            
            if result:
                sortie_value, bild_value, geometry_wkt, srid = result
                print(f"✓ Found image in database: sortie={sortie_value}, bildnummer={bild_value}")
                
                from shapely import wkt
                polygon = wkt.loads(geometry_wkt)
                
                # Use the CRS from the geometry
                crs_string = f'EPSG:{srid}' if srid else 'EPSG:25832'
                
                gdf = gpd.GeoDataFrame({
                    'sortie_ori': [sortie_value],
                    'Bildnummer': [bild_value],
                    'geometry': [polygon]
                }, crs=crs_string)
                
                print(f"  Geometry SRID: {srid}")
                print(f"  Polygon area: {polygon.area:.2f} square meters")
                return gdf
            else:
                print(f"✗ Image not found in database: sortie={sortie}, bildnummer={image_num}")
                return None
                
        except Exception as e:
            print(f"Error querying database: {e}")
            return None

    def clip_to_image_polygon(self, gdf, image_polygon_gdf):
        if gdf is None or gdf.empty:
            return None
        
        if image_polygon_gdf is None or image_polygon_gdf.empty:
            return None
        
        try:
            if gdf.crs != image_polygon_gdf.crs:
                print(f"CRS mismatch: gdf={gdf.crs}, polygon={image_polygon_gdf.crs}")
                # Try to project, but handle errors
                try:
                    image_polygon_gdf = image_polygon_gdf.to_crs(gdf.crs)
                except Exception as e:
                    print(f"Could not reproject: {e}")
                    # If reprojection fails, use the gdf's CRS as is
                    pass
            
            clipped = gpd.clip(gdf, image_polygon_gdf)
            
            if not clipped.empty:
                print(f"  Clipped successfully: {len(clipped)} features remaining")
                return clipped
            else:
                print("  No features remain after clipping")
                return None
                
        except Exception as e:
            print(f"Error clipping to image polygon: {e}")
            return None

    def safe_extract_value(self, val):
        """Safely extract a value from OSM attributes that might be lists, arrays, or NaN."""
        if isinstance(val, np.ndarray):
            if val.size == 0:
                return None
            if val.size > 0:
                try:
                    return val.flat[0]
                except:
                    return None
            return None
        
        if isinstance(val, pd.Series):
            if len(val) > 0:
                try:
                    return val.iloc[0]
                except:
                    return None
            return None
        
        if isinstance(val, (list, tuple)):
            if len(val) > 0:
                return val[0]
            return None
        
        if val is None:
            return None
        
        try:
            if pd.isna(val):
                return None
        except:
            if val is None:
                return None
        
        return val

    def safe_to_crs(self, gdf, target_crs):
        """Safely convert GeoDataFrame to target CRS with error handling."""
        try:
            return gdf.to_crs(target_crs)
        except Exception as e:
            print(f"  Warning: Could not convert to {target_crs}: {e}")
            print(f"  Using original CRS: {gdf.crs}")
            return gdf

    def get_osm_road_network(self, gdf_edges, gdf_extent):
        """
        Download OSM road network with only main roads, link roads, and minor roads.
        """
        try:
            # Try to convert to WGS84, but handle errors
            try:
                gdf_extent_wgs84 = gdf_extent.to_crs('EPSG:4326')
                bounds_wgs84 = gdf_extent_wgs84.total_bounds
            except Exception as e:
                print(f"  Warning: Could not convert to EPSG:4326: {e}")
                # Use the bounds directly if they're already in lat/lon
                bounds = gdf_extent.total_bounds
                # Assume the coordinates are in degrees if they're around these ranges
                if -180 <= bounds[0] <= 180 and -90 <= bounds[1] <= 90:
                    bounds_wgs84 = bounds
                else:
                    # Approximate conversion for UTM zone 32 (rough)
                    x, y = bounds[0], bounds[1]
                    lon = (x - 500000) / 111320
                    lat = y / 111320
                    bounds_wgs84 = (lon, lat, lon + 0.1, lat + 0.1)
                    print(f"  Using approximate bounds: {bounds_wgs84}")
            
            center_lat = (bounds_wgs84[1] + bounds_wgs84[3]) / 2
            center_lon = (bounds_wgs84[0] + bounds_wgs84[2]) / 2
            
            lat_dist = abs(bounds_wgs84[3] - bounds_wgs84[1]) * 111000
            lon_dist = abs(bounds_wgs84[2] - bounds_wgs84[0]) * 111000 * np.cos(np.radians(center_lat))
            dist_meters = max(lat_dist, lon_dist) / 2 * 1.1
            
            print(f"  OSM query center: ({center_lat}, {center_lon}), distance: {dist_meters:.0f} meters")
            
            # Define road filters - ONLY car roads
            main_road_filters = [
                'motorway', 'motorway_link', 'trunk', 'trunk_link',
                'primary', 'primary_link', 'secondary', 'secondary_link',
                'tertiary', 'tertiary_link'
            ]
            
            link_road_filters = [
                'motorway_link', 'trunk_link', 'primary_link',
                'secondary_link', 'tertiary_link', 'living_street'
            ]
            
            minor_road_filters = [
                'unclassified', 'residential', 'service', 
                'parking_aisle', 'driveway', 'alley',
                'road', 'construction'
            ]
            
            # Download OSM network
            try:
                G = ox.graph_from_point(
                    (center_lat, center_lon),
                    dist=dist_meters,
                    network_type='drive',
                    simplify=True,
                    retain_all=True
                )
            except Exception as e:
                print(f"  Error downloading OSM data: {e}")
                return None
            
            # Convert to GeoDataFrame
            gdf_osm_edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
            
            if gdf_osm_edges.empty:
                print("  No OSM data downloaded")
                return None
            
            # Safely extract highway values
            try:
                highway_values = []
                for idx, val in gdf_osm_edges['highway'].items():
                    extracted = self.safe_extract_value(val)
                    if extracted is None:
                        highway_values.append('unknown')
                    else:
                        highway_values.append(str(extracted))
                gdf_osm_edges['highway'] = highway_values
            except Exception as e:
                print(f"  Warning: Could not process highway column: {e}")
                gdf_osm_edges['highway'] = 'unknown'
            
            gdf_osm_edges['highway_type'] = gdf_osm_edges['highway'].astype(str)
            
            # Add road category columns
            gdf_osm_edges['category'] = 'unknown'
            
            main_road_mask = gdf_osm_edges['highway'].isin(main_road_filters)
            gdf_osm_edges.loc[main_road_mask, 'category'] = 'main_road'
            
            link_mask = gdf_osm_edges['highway'].isin(link_road_filters) & ~main_road_mask
            gdf_osm_edges.loc[link_mask, 'category'] = 'link_road'
            
            minor_mask = gdf_osm_edges['highway'].isin(minor_road_filters) & ~main_road_mask & ~link_mask
            gdf_osm_edges.loc[minor_mask, 'category'] = 'minor_road'
            
            # Filter out unknown categories
            gdf_osm_edges = gdf_osm_edges[gdf_osm_edges['category'] != 'unknown']
            
            if gdf_osm_edges.empty:
                print("  No car roads found in the downloaded OSM data")
                return None
            
            # Add road class
            gdf_osm_edges['road_class'] = 'other'
            
            motorway_mask = gdf_osm_edges['highway'].str.contains('motorway', na=False)
            gdf_osm_edges.loc[motorway_mask, 'road_class'] = 'motorway'
            
            primary_mask = gdf_osm_edges['highway'].str.contains('primary|trunk', na=False)
            gdf_osm_edges.loc[primary_mask & ~motorway_mask, 'road_class'] = 'primary'
            
            secondary_mask = gdf_osm_edges['highway'].str.contains('secondary', na=False)
            gdf_osm_edges.loc[secondary_mask & ~primary_mask & ~motorway_mask, 'road_class'] = 'secondary'
            
            tertiary_mask = gdf_osm_edges['highway'].str.contains('tertiary|residential|unclassified', na=False)
            gdf_osm_edges.loc[tertiary_mask & ~secondary_mask & ~primary_mask & ~motorway_mask, 'road_class'] = 'tertiary'
            
            # Project to the same CRS as edges
            try:
                gdf_osm_projected = gdf_osm_edges.to_crs(gdf_edges.crs)
            except Exception as e:
                print(f"  Warning: Could not project OSM data: {e}")
                gdf_osm_projected = gdf_osm_edges
            
            # Clip to extent
            try:
                gdf_osm_clipped = gpd.clip(gdf_osm_projected, gdf_extent)
            except Exception as e:
                print(f"  Warning: Could not clip OSM data: {e}")
                gdf_osm_clipped = gdf_osm_projected
            
            if gdf_osm_clipped.empty:
                print("  No car roads within extent")
                return None
            
            # Print statistics
            print(f"\n  Downloaded {len(gdf_osm_edges)} car road segments")
            print(f"  Clipped to {len(gdf_osm_clipped)} road segments")
            print(f"\n  Road Categories:")
            for category in ['main_road', 'link_road', 'minor_road']:
                count = len(gdf_osm_clipped[gdf_osm_clipped['category'] == category])
                if count > 0:
                    print(f"    {category}: {count} segments")
            
            return gdf_osm_clipped
            
        except Exception as e:
            print(f"  Error getting OSM data: {e}")
            import traceback
            traceback.print_exc()
            return None

    def calculate_intersection_with_clipping(self, gdf_edges, gdf_osm, image_polygon_gdf, 
                                            image_name):
        try:
            print(f"\nProcessing with image polygon clipping for {image_name}")
            
            # 1. Buffer edge lines
            print("  Buffering edge lines...")
            gdf_buffer_edges = gdf_edges.copy()
            gdf_buffer_edges['geometry'] = gdf_buffer_edges.buffer(self.buffer_edges)
            gdf_buffer_edges_dissolved = gdf_buffer_edges.dissolve()
            
            # Clip buffered edges to image polygon
            gdf_buffer_edges_clipped = self.clip_to_image_polygon(
                gdf_buffer_edges_dissolved, image_polygon_gdf
            )
            
            if gdf_buffer_edges_clipped is None or gdf_buffer_edges_clipped.empty:
                print("  No buffered edges within image polygon")
                return None
            
            # 2. Buffer OSM roads
            print("  Buffering OSM roads...")
            gdf_osm_buffered = gdf_osm.copy()
            gdf_osm_buffered['geometry'] = gdf_osm_buffered.buffer(self.buffer_osm)
            gdf_osm_buffered_dissolved = gdf_osm_buffered.dissolve()
            
            # Clip buffered OSM to image polygon
            gdf_osm_buffered_clipped = self.clip_to_image_polygon(
                gdf_osm_buffered_dissolved, image_polygon_gdf
            )
            
            if gdf_osm_buffered_clipped is None or gdf_osm_buffered_clipped.empty:
                print("  No buffered OSM roads within image polygon")
                return None
            
            # 3. Calculate intersection between clipped buffers
            print("  Calculating intersection...")
            gdf_intersection = gpd.overlay(
                gdf_buffer_edges_clipped, 
                gdf_osm_buffered_clipped, 
                how='intersection'
            )
            
            if gdf_intersection.empty:
                print("  No intersection found")
                return None
            
            # 4. Calculate areas and percentages
            buffer_edges_area = gdf_buffer_edges_clipped.geometry.area.sum()
            buffer_osm_area = gdf_osm_buffered_clipped.geometry.area.sum()
            intersection_area = gdf_intersection.geometry.area.sum()
            
            percentage = (intersection_area / buffer_osm_area) * 100 if buffer_osm_area > 0 else 0
            
            # 5. Save results
            intersection_output = self.output_dir / f"{Path(image_name).stem}_intersection_clipped.gpkg"
            gdf_intersection.to_file(intersection_output, driver='GPKG')
            
            buffer_edges_output = self.output_dir / f"{Path(image_name).stem}_buffer_edges_clipped.gpkg"
            gdf_buffer_edges_clipped.to_file(buffer_edges_output, driver='GPKG')
            
            buffer_osm_output = self.output_dir / f"{Path(image_name).stem}_buffer_osm_clipped.gpkg"
            gdf_osm_buffered_clipped.to_file(buffer_osm_output, driver='GPKG')
            
            # Save categorized OSM for analysis
            osm_categorized = self.output_dir / f"{Path(image_name).stem}_osm_categorized.gpkg"
            gdf_osm.to_file(osm_categorized, driver='GPKG')
            
            print(f"\n  Results for {image_name}:")
            print(f"    Buffered edges area (clipped): {buffer_edges_area:.2f} m²")
            print(f"    Buffered OSM area (clipped): {buffer_osm_area:.2f} m²")
            print(f"    Intersection area (clipped): {intersection_area:.2f} m²")
            print(f"    Percentage (Intersection / Buffered OSM): {percentage:.2f}%")
            
            result = {
                'image_name': image_name,
                'buffer_edges_area': buffer_edges_area,
                'buffer_osm_area': buffer_osm_area,
                'intersection_area': intersection_area,
                'percentage': percentage,
                'features_count': len(gdf_intersection),
                'osm_categories': gdf_osm['category'].value_counts().to_dict(),
                'osm_road_classes': gdf_osm['road_class'].value_counts().to_dict()
            }
            
            return result
            
        except Exception as e:
            print(f"Error calculating intersection with clipping: {e}")
            return None

    def parse_image_info(self, filename):
        filename = filename.replace('.lnk', '').replace(' - Verknüpfung', '').replace('_Finalized', '')
        parts = [p.strip() for p in filename.split('#') if p.strip()]
        
        image_num = flight_num = sortie = bestell_id = date = None
        
        if parts:
            image_num = parts[0]
        
        for part in parts:
            if part.startswith('ID') and any(c.isdigit() for c in part[2:]):
                numbers = re.findall(r'\d+', part)
                bestell_id = numbers[0] if numbers else None
                break
        
        for part in parts:
            if re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                date = part
                break
        
        if len(parts) >= 3:
            for part in parts[1:-1]:
                if part != bestell_id and not re.search(r'\d{1,2}[._]\d{1,2}[._]\d{2,4}', part):
                    if not flight_num:
                        flight_num = part
                    elif not sortie:
                        sortie = part
        
        if image_num:
            image_num = re.sub(r'_[a-zA-Z_]+.*$', '', image_num)
        
        return {
            'image_number': image_num,
            'flight_number': flight_num,
            'sortie': sortie,
            'bestell_id': bestell_id,
            'date': date,
            'original_name': filename
        }

    def find_and_process_shortcuts(self):
        print(f"\nScanning for shortcut files in {self.source_path}...")
        
        for root, dirs, files in os.walk(self.source_path):
            for file in files:
                if file.lower().endswith('.lnk'):
                    file_without_ext = file[:-4]
                    parsed_info = self.parse_image_info(file_without_ext)
                    parsed_info['full_path'] = Path(root) / file
                    parsed_info['folder'] = root
                    self.shortcuts.append(parsed_info)
        
        print(f"Found {len(self.shortcuts)} shortcut files")
        return self.shortcuts

    def get_shortcut_target(self, lnk_path):
        try:
            pythoncom.CoInitialize()
            shell = win32com.client.Dispatch("WScript.Shell")
            shortcut = shell.CreateShortCut(str(lnk_path))
            target_path = shortcut.TargetPath
            pythoncom.CoUninitialize()
            return target_path if target_path else None
        except Exception as e:
            print(f"Error reading shortcut {lnk_path}: {e}")
            return None

    def find_modified_versions(self, original_path):
        if not original_path or not original_path.exists():
            return []
        
        folder = original_path.parent
        stem = original_path.stem
        modified_patterns = ['_modified', '_modifiziert', '_georeferenced', '_processed']
        modified_files = []
        
        for pattern in modified_patterns:
            potential_file = folder / f"{stem}{pattern}{original_path.suffix}"
            if potential_file.exists():
                modified_files.append(potential_file)
        
        for file in folder.iterdir():
            if file.is_file() and original_path.suffix == file.suffix:
                file_stem = file.stem
                if (file_stem.startswith(stem) and 
                    any(pattern in file_stem for pattern in modified_patterns)):
                    modified_files.append(file)
        
        return list(set(modified_files))

    def extract_shortcut_targets(self):
        print("\nExtracting shortcut targets and finding modified versions...")
        
        for shortcut_data in self.shortcuts:
            lnk_path = shortcut_data['full_path']
            target_path = self.get_shortcut_target(lnk_path)
            
            if target_path:
                target_path = Path(target_path)
                shortcut_data.update({
                    'target_path': target_path,
                    'target_exists': target_path.exists()
                })
                
                if shortcut_data['target_exists']:
                    modified_files = self.find_modified_versions(target_path)
                    if modified_files:
                        tiff_files = [f for f in modified_files if f.suffix.lower() in ['.tif', '.tiff']]
                        if tiff_files:
                            shortcut_data['modified_files'] = tiff_files
                            self.shortcuts_with_targets.append(shortcut_data)
        
        print(f"Found {len(self.shortcuts_with_targets)} shortcuts with modified TIFF files")
        return self.shortcuts_with_targets

    def resample_to_2m(self, input_path, output_path):
        try:
            with rasterio.open(input_path) as src:
                target_res = 2.0
                left, bottom, right, top = src.bounds
                width = int((right - left) / target_res)
                height = int((top - bottom) / target_res)
                
                transform, width, height = calculate_default_transform(
                    src.crs, src.crs, src.width, src.height, 
                    left, bottom, right, top, resolution=target_res
                )
                
                kwargs = src.meta.copy()
                kwargs.update({
                    'transform': transform,
                    'width': width,
                    'height': height,
                    'res': (target_res, target_res)
                })
                
                with rasterio.open(output_path, 'w', **kwargs) as dst:
                    for i in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, i),
                            destination=rasterio.band(dst, i),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=src.crs,
                            resampling=Resampling.bilinear
                        )
                
                return output_path
        except Exception as e:
            print(f"Error resampling {input_path.name}: {e}")
            return None

    def extract_edges(self, image_path):
        image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image is None:
            print(f"Failed to read image: {image_path}")
            return None
        
        blurred = cv2.GaussianBlur(image, (5, 5), 0)
        edges = cv2.Canny(blurred, 50, 150, apertureSize=3)
        
        dataset = gdal.Open(str(image_path))
        if not dataset:
            print(f"Failed to open {image_path} for geotransform")
            return None
        
        geotransform = dataset.GetGeoTransform()
        projection = dataset.GetProjection()
        
        if projection:
            srs = osr.SpatialReference()
            srs.ImportFromWkt(projection)
            crs_used = srs.ExportToWkt()
        else:
            crs_used = 'EPSG:25832'
        
        x_origin = geotransform[0]
        y_origin = geotransform[3]
        pixel_width = geotransform[1]
        pixel_height = geotransform[5]
        
        def pixel_to_geo(pixel_x, pixel_y):
            return x_origin + pixel_x * pixel_width, y_origin + pixel_y * pixel_height
        
        contours, _ = cv2.findContours(edges, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        
        lines = []
        for contour in contours:
            if len(contour) >= 2:
                coords = contour.squeeze().tolist()
                if isinstance(coords[0], list):
                    geo_coords = [pixel_to_geo(p[0], p[1]) for p in coords]
                    line = LineString(geo_coords)
                    if not line.is_empty and line.length > 0:
                        lines.append(line)
        
        dataset = None
        
        if lines:
            gdf = gpd.GeoDataFrame({'geometry': lines}, crs=crs_used)
            return gdf
        else:
            return None

    def process_image(self, shortcut_data, image_polygon_gdf=None):
        try:
            modified_files = shortcut_data.get('modified_files', [])
            if not modified_files:
                return None
            
            original_image = modified_files[0]
            image_name = original_image.name
            
            sortie = shortcut_data.get('sortie')
            image_num = shortcut_data.get('image_number')
            
            if sortie and image_num:
                print(f"\nDatabase status for {image_name}:")
                if image_polygon_gdf is not None and not image_polygon_gdf.empty:
                    print(f"  ✓ Image found in database with geometry")
                    print(f"  ✓ Polygon area: {image_polygon_gdf.geometry.area.iloc[0]:.2f} m²")
                else:
                    print(f"  ✗ Image not found in database or no geometry")
            else:
                print(f"  ℹ No sortie/image_num found in filename")
            
            resampled_path = self.output_dir / f"{Path(image_name).stem}_resampled_2m{Path(image_name).suffix}"
            resampled_file = self.resample_to_2m(original_image, resampled_path)
            
            if not resampled_file:
                return None
            
            gdf_edges = self.extract_edges(resampled_file)
            if gdf_edges is None or gdf_edges.empty:
                print(f"  No edges extracted from {image_name}")
                return None
            
            edges_output = self.output_dir / f"{Path(image_name).stem}_edges.gpkg"
            gdf_edges.to_file(edges_output, driver='GPKG')
            print(f"  ✓ Saved edges: {len(gdf_edges)} polylines")
            
            bounds = gdf_edges.total_bounds
            extent_polygon = box(bounds[0], bounds[1], bounds[2], bounds[3])
            gdf_extent = gpd.GeoDataFrame({'geometry': [extent_polygon]}, crs=gdf_edges.crs)
            
            # Get OSM road network (only car roads)
            gdf_osm_clipped = self.get_osm_road_network(gdf_edges, gdf_extent)
            if gdf_osm_clipped is None:
                print(f"  No OSM car road data available for {image_name}")
                return None
            
            osm_output = self.output_dir / f"{Path(image_name).stem}_osm_clipped.gpkg"
            gdf_osm_clipped.to_file(osm_output, driver='GPKG')
            
            # Calculate intersection with clipping (using pre-set buffer distances)
            result = self.calculate_intersection_with_clipping(
                gdf_edges, gdf_osm_clipped, image_polygon_gdf, image_name
            )
            
            if result:
                result['sortie'] = sortie
                result['image_number'] = image_num
                result['edges_count'] = len(gdf_edges)
                result['osm_count'] = len(gdf_osm_clipped)
                return result
            else:
                return None
                
        except Exception as e:
            print(f"Error processing {shortcut_data.get('target_filename', 'unknown')}: {e}")
            return None

    def run(self):
        print("\n" + "=" * 80)
        print("GEOREF EDGE PROCESSOR - STARTING PROCESSING")
        print("=" * 80)
        
        # Get buffer distances once for all images
        self.get_buffer_distances()
        
        if not self.connect_database():
            print("Warning: Continuing without database connection")
        
        try:
            self.find_and_process_shortcuts()
            self.extract_shortcut_targets()
            
            if not self.shortcuts_with_targets:
                print("No valid shortcuts with modified TIFF files found.")
                return
            
            print(f"\nProcessing {len(self.shortcuts_with_targets)} images...")
            
            for i, shortcut in enumerate(self.shortcuts_with_targets, 1):
                print(f"\n{'='*60}")
                print(f"Processing image {i}/{len(self.shortcuts_with_targets)}")
                print(f"{'='*60}")
                
                sortie = shortcut.get('sortie')
                image_num = shortcut.get('image_number')
                image_polygon_gdf = None
                
                if sortie and image_num and self.db_connection:
                    image_polygon_gdf = self.get_image_polygon(sortie, image_num)
                else:
                    if not sortie:
                        print(f"  Warning: No sortie found in filename")
                    if not image_num:
                        print(f"  Warning: No image number found in filename")
                
                result = self.process_image(shortcut, image_polygon_gdf)
                if result:
                    self.all_results.append(result)
            
            # Print summary table
            self.print_results_table()
            
        finally:
            self.close_database()
        
        print(f"\nOutput directory: {self.output_dir}")
        print("Processing completed!")

    def print_results_table(self):
        """Print a formatted summary table of all results."""
        print("\n" + "=" * 120)
        print("FINAL RESULTS SUMMARY")
        print("=" * 120)
        print(f"Buffer distances used:")
        print(f"  Edge buffer: {self.buffer_edges} meters")
        print(f"  OSM buffer: {self.buffer_osm} meters")
        print("=" * 120)
        
        if not self.all_results:
            print("\nNo results to display.")
            return
        
        # Create DataFrame for better display
        df_data = []
        for r in self.all_results:
            # Get category counts
            categories = r.get('osm_categories', {})
            main_road = categories.get('main_road', 0)
            link_road = categories.get('link_road', 0)
            minor_road = categories.get('minor_road', 0)
            
            df_data.append({
                'Image': Path(r['image_name']).stem[:30] + '...' if len(Path(r['image_name']).stem) > 30 else Path(r['image_name']).stem,
                'Sortie': r.get('sortie', 'N/A'),
                'Bildnr': r.get('image_number', 'N/A'),
                'Edges': r.get('edges_count', 0),
                'OSM Segments': r.get('osm_count', 0),
                'Main Roads': main_road,
                'Link Roads': link_road,
                'Minor Roads': minor_road,
                'Edge Buffer (m²)': f"{r.get('buffer_edges_area', 0):.0f}",
                'OSM Buffer (m²)': f"{r.get('buffer_osm_area', 0):.0f}",
                'Intersection (m²)': f"{r.get('intersection_area', 0):.0f}",
                'Percentage (%)': f"{r.get('percentage', 0):.2f}"
            })
        
        df = pd.DataFrame(df_data)
        
        # Print the table
        print("\n" + df.to_string(index=False))
        
        # Print summary statistics
        print("\n" + "-" * 120)
        print("SUMMARY STATISTICS:")
        print("-" * 120)
        
        if self.all_results:
            total_edges = sum([r.get('edges_count', 0) for r in self.all_results])
            total_osm = sum([r.get('osm_count', 0) for r in self.all_results])
            avg_percentage = np.mean([r.get('percentage', 0) for r in self.all_results])
            
            # Calculate totals by category
            total_main = sum([r.get('osm_categories', {}).get('main_road', 0) for r in self.all_results])
            total_link = sum([r.get('osm_categories', {}).get('link_road', 0) for r in self.all_results])
            total_minor = sum([r.get('osm_categories', {}).get('minor_road', 0) for r in self.all_results])
            
            print(f"  Total images processed: {len(self.all_results)}")
            print(f"  Total edge polylines extracted: {total_edges}")
            print(f"  Total OSM road segments: {total_osm}")
            print(f"    Main roads: {total_main}")
            print(f"    Link roads: {total_link}")
            print(f"    Minor roads: {total_minor}")
            print(f"  Average intersection percentage: {avg_percentage:.2f}%")
            print(f"  Min percentage: {min([r.get('percentage', 0) for r in self.all_results]):.2f}%")
            print(f"  Max percentage: {max([r.get('percentage', 0) for r in self.all_results]):.2f}%")
            
            # Show best and worst performers
            best = max(self.all_results, key=lambda x: x.get('percentage', 0))
            worst = min(self.all_results, key=lambda x: x.get('percentage', 0))
            
            print(f"\n  Best performer: {Path(best['image_name']).stem}")
            print(f"    Percentage: {best.get('percentage', 0):.2f}%")
            print(f"  Worst performer: {Path(worst['image_name']).stem}")
            print(f"    Percentage: {worst.get('percentage', 0):.2f}%")
        
        print("=" * 120)

# Main execution
if __name__ == "__main__":
    SOURCE_PATH = r"W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked"
    OUTPUT_DIR = r"D:\FBM try\outputs\Vectorization & morphological filtering\OUTPUTS"
    
    processor = GeorefEdgeProcessor(SOURCE_PATH, OUTPUT_DIR)
    processor.run()

QGIS_ROOT: C:\Program Files\QGIS 3.34.12
PROJ_DIR: C:\Program Files\QGIS 3.34.12\share\proj
GDAL_DIR: C:\Program Files\QGIS 3.34.12\share\gdal
proj.db exists: True
pyproj data dir set to: C:\Program Files\QGIS 3.34.12\share\proj
GDAL PROJ config set
Initialized GeorefEdgeProcessor
Source path: W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked
Output directory: D:\FBM try\outputs\Vectorization & morphological filtering\OUTPUTS
✓ PROJ working: EPSG:4326 is valid

GEOREF EDGE PROCESSOR - STARTING PROCESSING

BUFFER DISTANCE CONFIGURATION
These buffer distances will be applied to ALL images.

Buffer distances set:
  Edge buffer: 10.0 meters
  OSM buffer: 10.0 meters
Successfully connected to the database

Scanning for shortcut files in W:\_Vorbereitung Digitalisierung\Bilder ohne Geometrie\GEOREFERENCED\1. HarmonyMarked...
Found 28 shortcut files

Extracting shortcut targets and finding modified versions...


Win32 exception occurred releasing IUnknown at 0x0000012CA4B2BF20
Win32 exception occurred releasing IUnknown at 0x0000012CA4B2BF80


Found 28 shortcuts with modified TIFF files

Processing 28 images...

Processing image 1/28
Using columns: sortie='sortie_ori', bild='Bildnummer', geom='geom'
Querying database with sortie='2_3674', bildnummer='3242'
✓ Found image in database: sortie=2_3674, bildnummer=3242
  Geometry SRID: 25832
  Polygon area: 9434998.01 square meters

Database status for 3242#_#2_3674#ID1737#23_07_1947_modified.tif:
  ✓ Image found in database with geometry
  ✓ Polygon area: 9434998.01 m²
  ✓ Saved edges: 450 polylines
  OSM query center: (49.49104757602605, 9.807524249517254), distance: 2565 meters

  Downloaded 1585 car road segments
  Clipped to 1344 road segments

  Road Categories:
    main_road: 179 segments
    link_road: 54 segments
    minor_road: 1111 segments

Processing with image polygon clipping for 3242#_#2_3674#ID1737#23_07_1947_modified.tif
  Buffering edge lines...
  Clipped successfully: 1 features remaining
  Buffering OSM roads...
  Clipped successfully: 1 features remaining
  C